# Qwen3 Fine-Tuned for Mutual Fund Tool Calling
### Teaching a small language model to decide which tool to call, with which values, on a free Colab T4 GPU

---

## The idea in one paragraph

We build the "brain" of a small **mutual fund assistant**. A user types a request such as *"Start a SIP of ₹5,000 in Axis Children's Fund on the 10th"*. The model does **not** do the work itself. It **decides** which of our three Python functions (called **tools**) should run, and fills in the values that function needs. Our own Python code then **runs** that function and shows the result. We fine-tune the model so that it makes this decision reliably, in exactly the format our code can read.

---

## The pieces

### The model

**Qwen3-1.7B**, a small open model from Alibaba's Qwen team.

- 1.7 billion parameters.
- Apache 2.0 licence, so free to use, and no login needed.
- It already knows a general tool-calling format. We train it to use **our** three tools precisely.

### The three tools

These are normal Python functions that **we** write in Part 2. The model never sees their code. It only sees a short description of each one.

| Tool | What it does | Values it needs (arguments) | Where its data comes from |
|---|---|---|---|
| `get_nav` | Returns the latest NAV (price of one unit) of a mutual fund | `fund_name` | **Real:** AMFI's public daily NAV file |
| `get_portfolio` | Returns the funds a customer holds | `customer_id` | Made-up demo customers C1001 to C1005 |
| `start_sip` | Starts a monthly SIP and returns a confirmation ID | `fund_name`, `amount`, `day_of_month` | Made-up: recorded in a list in this notebook, nothing real happens |

### What happens when you type a request

```
1. YOU TYPE      "Start a SIP of ₹5,000 in Axis Children's Fund on the 10th"
                        │
                        ▼
2. THE MODEL     reads your request + the descriptions of the 3 tools
   DECIDES       and writes only this:
                 <tool_call>
                 {"name": "start_sip", "arguments": {"fund_name": "Axis Children's Fund", "amount": 5000, "day_of_month": 10}}
                 </tool_call>
                        │
                        ▼
3. OUR CODE      reads that text, finds the tool name "start_sip",
   RUNS          and runs our Python function start_sip("Axis Children's Fund", 5000, 10)
                        │
                        ▼
4. THE TOOL      {"status": "created", "sip_id": "SIP-0001", ...}
   RETURNS
                        │
                        ▼
5. THE MODEL     turns the result into a friendly sentence:
   REPLIES       "Your SIP of ₹5,000 in Axis Children's Fund is set up for the 10th of every month."
```

**The model decides. Our code does.**

Not every request needs a tool:

| You type | What the model should do |
|---|---|
| "What is the NAV of SBI Credit Risk Fund?" | Call `get_nav` with `fund_name = "SBI Credit Risk Fund"` |
| "Show my holdings, customer ID C1003" | Call `get_portfolio` with `customer_id = "C1003"` |
| "What is an SIP?" | **No tool.** Answer in plain text |
| "Start a SIP in Axis Children's Fund" | **No tool yet.** The amount and day are missing, so ask the user for them |

### The training data

There is no ready-made dataset for our exact three tools, so the notebook **generates** it in Part 4, in a few seconds:

- **Sentence patterns:** we write several ways of asking for each tool, for example "What is the NAV of {fund}?" and "{fund} ka NAV kya hai?".
- **Real fund names:** we fill in real scheme names taken from the AMFI file, plus random amounts, days and customer IDs.
- **Correct answers:** because we chose the values ourselves, we know the correct answer for every sentence, and write it automatically.

| Kind of example | Share of the data |
|---|---|
| Needs `get_nav` | 27% |
| Needs `start_sip` | 27% |
| Needs `get_portfolio` | 16% |
| A detail is missing, so the model should ask | 16% |
| General question, so no tool is needed | 14% |

By default: **1,000 training examples** and **100 checking examples**. The checking examples use **sentence patterns and fund names the model never saw in training**, so the score is honest.

### How we know it worked: the scores

We score the model on the 100 checking examples **before** and **after** training:

| Score | Question it answers |
|---|---|
| Readable tool call | When a tool was needed, could our code read the model's output? |
| Correct tool | Did it pick the right tool? |
| Correct tool and arguments | Did it pick the right tool **and** fill in every value exactly right (for example the amount as the number 5000)? |
| No tool when none needed | For general questions, did it answer without calling a tool? |
| Asked for missing details | When a detail was missing, did it ask instead of guessing? |
| Overall correct | All checking examples together |

---

## The journey, part by part

| Part | What we do | What comes out |
|---|---|---|
| **0. Key ideas** | Read: what function calling is, why we fine-tune, what "training only on the answer" means | The ideas for the rest of the notebook |
| **1. Setup** | Check the GPU, install libraries, set all settings in one cell | A ready Colab session |
| **2. The three tools** | Download the real AMFI NAV file, write the 3 tool functions, write their descriptions, test them by hand | Working tools, before any AI is involved |
| **3. Load Qwen3** | Load the tokenizer, see the exact prompt the model receives, load the model in 4-bit | The model on the GPU |
| **4. Training data** | Write sentence patterns, fill them with real fund names, build 1,000 training and 100 checking examples | Two lists of examples with known correct answers |
| **5. Before** | Ask the untouched model all 100 checking requests and score it | The "before" scores |
| **6. Prepare for training** | Turn each example into token IDs, and mark which tokens the model should learn | Training-ready data |
| **7. LoRA** | Freeze the model and attach small trainable tables | About 1% of the numbers can be trained |
| **8. Train** | Train and chart the loss | A trained model |
| **9. After** | Score again and compare with "before" | The before/after table |
| **10. Live assistant** | Type your own request: model decides → tool runs → model replies | A working demo |
| **11. Save** | Save the adapter and load it back | An adapter folder of about 80 MB |
| **12. Recap** | Summary, limits, exercises | |

---

## What you need to do

1. In the Colab menu choose **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the cells **from top to bottom, one at a time** (Shift + Enter).
3. Three cells take several minutes: scoring "before" (Part 5), training (Part 8) and scoring "after" (Part 9). My estimates: about 5 minutes for each scoring run and roughly 15–20 minutes for training. **These are estimates, not yet measured on a T4.**
4. In **Part 10**, type your own requests into the box and watch the model choose tools.

Look for these as you go:

- **Predict first:** guess the result before running the cell.
- **Try it:** change something small and run the cell again.
- **Check yourself:** a question with a hidden answer you can click to open.

Every code cell starts with a block explaining what the cell does, and every line of code has its own explanation above it.

---
# Part 0 — Key ideas before we start

## 0.1 What "function calling" (tool calling) means

A language model can only produce text. It cannot look up today's NAV or create a SIP on its own. **Function calling** connects it to code that can:

1. We describe our functions to the model: name, what it does, and which values it needs.
2. When a request needs one of them, the model writes out **which function** and **which values**, in a fixed format (JSON).
3. Our code reads that, runs the real function, and gets a real result.

So the model's job is only to **choose and fill in**. Running the function is our code's job.

## 0.2 Why fine-tune, if Qwen3 already knows tool calling?

Qwen3 was trained to use tools in general. But in our situation, small models typically make these mistakes:

| Mistake | Example |
|---|---|
| Wrong value type | `"amount": "5000 rupees"` instead of `"amount": 5000` |
| Value not converted | Leaves `"5k"` or `"₹5,000"` as text instead of the number 5000 |
| Fund name changed | Writes "Axis Childrens Fund" instead of copying "Axis Children's Fund" exactly |
| Calls a tool when none is needed | Calls `get_nav` for "What does NAV mean?" |
| Guesses missing details | Calls `start_sip` with a made-up amount, instead of asking the user |

Fine-tuning on our own examples teaches the model **our** rules for **our** tools.

## 0.3 Fine-tuning teaches behaviour, tools supply facts

The training data contains **no NAV values at all**. It only contains requests and the correct tool choice. The NAV is fetched **live** from AMFI each time `get_nav` runs.

If we trained NAVs into the model, they would be wrong the next day. With a tool, they are always current.

## 0.4 Training only on the answer

Every training example has two parts:

```
PART A (the question):  system instruction + the 3 tool descriptions + the user's request   ← about 500 tokens, mostly identical in every example
PART B (the answer):    <tool_call>{"name": "get_nav", "arguments": {...}}</tool_call>       ← about 40 tokens, what we want the model to learn
```

We tell the trainer to **only measure and learn from Part B**. Part A is still read by the model, but its tokens are marked "ignore" when the loss is calculated.

Why: Part A is almost the same in every example. If it counted, most of the loss would come from repeating the same tool descriptions, and the loss number would tell us little about the answers. (In the TinyLlama notebook we trained on the whole text, and listed this as a possible improvement. Here we use it.)

## 0.5 QLoRA, in one line each

- **Q (quantization):** store the model's numbers in 4 bits instead of 32, so it fits on the GPU.
- **LoRA:** freeze the model and train small extra tables next to it, about 1% of its size.

Both are explained in detail in the TinyLlama notebook. Here they work exactly the same way.

---
# Part 1 — Setup

## What we are doing in this part

1. **1.1** Confirm a GPU is switched on.
2. **1.2** Install the libraries and import the tools we need.
3. **1.3** Put every setting in one cell, with sliders you can change.

At the end of Part 1 nothing has been downloaded or trained yet.

## 1.1 Is the GPU switched on?

In [1]:
# =============================================================================
# CELL: CHECK THAT COLAB HAS GIVEN US A GPU
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Training a model means doing billions of multiplications.
# A GPU (graphics card) does these far faster than the normal processor.
#
# This cell asks one question: "Is there a GPU?"
#   - If YES -> it prints the GPU's name and its memory.
#               On Colab this should be "Tesla T4" with about 15 GB.
#   - If NO  -> it prints how to switch the GPU on.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: import torch
# -----------------------------------------------------------------------------
# - Loads PyTorch, the library that stores the model's numbers and does all
#   the maths on them. In code it is called "torch".
# - Colab already has it installed.
import torch


# -----------------------------------------------------------------------------
# LINE 2: is a GPU available?
# -----------------------------------------------------------------------------
# - "CUDA" is NVIDIA's system that lets programs use an NVIDIA GPU (the T4 is
#   one).
# - torch.cuda.is_available() gives back True if a GPU can be used.
if torch.cuda.is_available():

    # The name of GPU number 0 (computers count from 0; Colab gives one GPU).
    gpu_name = torch.cuda.get_device_name(0)

    # Its total memory in bytes, divided by 1024 x 1024 x 1024 to get GB.
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

    # Show both. {gpu_memory_gb:.1f} shows 1 digit after the decimal point.
    print(f"GPU found: {gpu_name} with {gpu_memory_gb:.1f} GB of memory")

# -----------------------------------------------------------------------------
# LINE 3: otherwise
# -----------------------------------------------------------------------------
else:

    # Tell the user how to switch the GPU on. Nothing crashes; it is a warning.
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

GPU found: Tesla T4 with 14.6 GB of memory


## 1.2 Install the libraries

| Library | Its job in this notebook |
|---|---|
| `transformers` | Loads Qwen3 and its tokenizer, and runs training |
| `peft` | Adds LoRA (the small trainable tables) |
| `bitsandbytes` | Loads the model in 4-bit form |
| `datasets` | Holds the training examples in the format the trainer expects |
| `accelerate` | Places the model on the GPU |

Qwen3 needs `transformers` version 4.51 or newer. The install cell upgrades to the newest version. Red warnings about other packages (for example `cudf` or `pyarrow`) are normal on Colab and do not affect this notebook.

In [2]:
# =============================================================================
# CELL: INSTALL THE LIBRARIES
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Downloads and installs five Python libraries into this Colab session.
# They stay installed only until the session ends.
#
# Reading the command:
#   !        -> run this line as a terminal command, not as Python
#   pip      -> Python's installer
#   install  -> install the libraries listed after it
#   -q       -> "quiet": print less
#   -U       -> "upgrade": replace older versions with the newest
#
# A red "ERROR: pip's dependency resolver..." message about other packages
# (such as cudf or pyarrow) is about packages we do not use. Safe to ignore.
# =============================================================================

!pip install -q -U transformers peft bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


In [3]:
# =============================================================================
# CELL: IMPORT THE TOOLS WE NEED
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Installing a library puts it on the computer. IMPORTING it makes it usable
# in this notebook.
#   import X          -> bring in the whole library, use it as X.something
#   from X import Y   -> bring in only Y, use it as Y
#
# At the end it prints the versions of the main libraries. If something
# fails later, these versions help when searching for the error.
# =============================================================================


# -----------------------------------------------------------------------------
# Python's built-in tools (nothing to install)
# -----------------------------------------------------------------------------
import json  # reads and writes JSON text. The model's tool calls are JSON.
import os  # works with files and folders. Used in Part 11 to list saved files.
import random  # makes random choices. Used in Part 4 to generate examples.
import re  # "regular expressions": finds patterns in text. Used to find <tool_call> in the model's output.
import time  # measures how long something takes. Used to time the scoring runs.

# -----------------------------------------------------------------------------
# Libraries that come with Colab
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt  # draws charts. Used for the loss chart in Part 8.
import pandas as pd  # shows tables neatly. Used for the before/after scores in Part 9.
import requests  # downloads files from the internet. Used to download the AMFI NAV file.

# -----------------------------------------------------------------------------
# The libraries we just installed
# -----------------------------------------------------------------------------
import datasets  # holds training data; imported here to print its version
import peft  # LoRA; imported here to print its version
import transformers  # models and training; imported here to print its version

# Dataset: the table format the trainer expects. Used in Part 6.
from datasets import Dataset

# The LoRA tools (Part 7 and Part 11):
#   LoraConfig                      -> holds the LoRA settings
#   PeftModel                       -> loads a saved adapter back onto a model
#   get_peft_model                  -> attaches the LoRA tables to a model
#   prepare_model_for_kbit_training -> gets a 4-bit model ready for training
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

# The model and training tools:
from transformers import (
    AutoModelForCausalLM,    # loads a text-generating model by name (Part 3)
    AutoTokenizer,           # loads the tokenizer that belongs to the model (Part 3)
    BitsAndBytesConfig,      # holds the 4-bit compression settings (Part 3)
    DataCollatorForSeq2Seq,  # builds batches and pads the "learn / ignore" labels (Part 8)
    Trainer,                 # the ready-made training loop (Part 8)
    TrainingArguments,       # the training settings (Part 8)
)


# -----------------------------------------------------------------------------
# Show the versions
# -----------------------------------------------------------------------------
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)  # must be 4.51 or newer for Qwen3
print("peft        ", peft.__version__)
print("datasets    ", datasets.__version__)

torch        2.11.0+cu130
transformers 5.19.0
peft         0.21.2
datasets     5.1.0


## 1.3 All the settings in one place

Every value you might want to change is in the next cell. In Colab, the sliders and boxes appear on the right side of the cell. Change a value, then re-run the notebook **from this cell onwards**.

| Setting | Default | What it controls |
|---|---|---|
| `NUM_TRAIN_EXAMPLES` | 1000 | How many examples the model learns from |
| `NUM_CHECK_EXAMPLES` | 100 | How many examples are kept aside for scoring. Use 50 for a faster live demo |
| `MAX_TOKENS` | 768 | Longest example allowed, in tokens |
| `LORA_R` | 16 | The LoRA rank: how many trainable numbers LoRA adds |
| `LEARNING_RATE` | 0.0002 | How big each adjustment is during training |
| `NUM_EPOCHS` | 1 | How many times the model goes through the training examples |

In [4]:
# =============================================================================
# CELL: ALL THE SETTINGS IN ONE PLACE
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Creates one variable for every setting the notebook uses. Nothing is
# downloaded or trained here.
#
# "#@param {...}" at the end of a line is only a comment to Python, but
# Colab reads it and draws a slider or input box on the right of the cell.
# It must stay on the same line as its variable.
#
# Names in CAPITAL LETTERS are a Python habit for "a setting that should not
# change while the notebook runs".
# =============================================================================


# -----------------------------------------------------------------------------
# MODEL_ID: which model we fine-tune
# -----------------------------------------------------------------------------
# - The model's name on huggingface.co: account "Qwen", model "Qwen3-1.7B".
# - 1.7B = 1.7 billion parameters.
MODEL_ID = "Qwen/Qwen3-1.7B"

# -----------------------------------------------------------------------------
# AMFI_URL: where the real NAV data comes from
# -----------------------------------------------------------------------------
# - AMFI (Association of Mutual Funds in India) publishes one text file with
#   the latest NAV of every mutual fund scheme in India, updated every
#   business day.
# - The get_nav tool reads this file. It is also our source of real fund
#   names for the training data.
AMFI_URL = "https://portal.amfiindia.com/spages/NAVAll.txt"

# -----------------------------------------------------------------------------
# NUM_TRAIN_EXAMPLES: how many examples the model learns from
# -----------------------------------------------------------------------------
# - More examples usually means a better result, and a longer training time.
# - Slider from 400 to 3,000, in steps of 200.
NUM_TRAIN_EXAMPLES = 1000  #@param {type:"slider", min:400, max:3000, step:200}

# -----------------------------------------------------------------------------
# NUM_CHECK_EXAMPLES: how many examples are kept aside for scoring
# -----------------------------------------------------------------------------
# - The model never learns from these. We score it on them before and after
#   training.
# - Each one is answered by the model one at a time, so a smaller number
#   makes Parts 5 and 9 faster. 50 is fine for a live seminar.
NUM_CHECK_EXAMPLES = 100  #@param {type:"slider", min:50, max:300, step:50}

# -----------------------------------------------------------------------------
# MAX_TOKENS: the longest example we allow
# -----------------------------------------------------------------------------
# - Our examples are about 550 tokens each (most of it is the tool
#   descriptions). 768 leaves room and drops nothing in practice.
MAX_TOKENS = 768  #@param {type:"slider", min:512, max:1024, step:128}

# -----------------------------------------------------------------------------
# LORA_R: the LoRA rank
# -----------------------------------------------------------------------------
# - How many patterns each LoRA change table is built from.
# - Bigger -> more trainable numbers, more freedom. Smaller -> fewer.
LORA_R = 16  #@param {type:"slider", min:4, max:64, step:4}

# -----------------------------------------------------------------------------
# LEARNING_RATE: the size of each adjustment during training
# -----------------------------------------------------------------------------
# - 2e-4 means 2 x 10^-4 = 0.0002, a common starting value for LoRA.
LEARNING_RATE = 2e-4  #@param {type:"number"}

# -----------------------------------------------------------------------------
# NUM_EPOCHS: how many passes over the training examples
# -----------------------------------------------------------------------------
NUM_EPOCHS = 1  #@param {type:"slider", min:1, max:3, step:1}

# -----------------------------------------------------------------------------
# ADAPTER_DIR: the folder where the trained LoRA tables are saved (Part 11)
# -----------------------------------------------------------------------------
ADAPTER_DIR = "qwen3-mf-tool-calling-lora"

# -----------------------------------------------------------------------------
# SEED: fixes all random choices
# -----------------------------------------------------------------------------
# - The data generator and the trainer make random choices. Fixing the seed
#   means everyone who runs this notebook gets the same examples and a
#   very similar result. 42 is just a commonly used number.
SEED = 42

# -----------------------------------------------------------------------------
# SYSTEM_PROMPT: the standing instruction given to the model every time
# -----------------------------------------------------------------------------
# - It sits above every request, in training AND when we ask questions.
#   It must be identical in both.
# - It states our three rules: use a tool when needed, ask when a detail is
#   missing, answer general questions directly.
SYSTEM_PROMPT = (
    "You are a mutual fund assistant for an Indian investment app. "
    "Call a tool when the user needs a fund's NAV, their portfolio, or wants to start a SIP. "
    "If a detail the tool needs is missing, ask the user for it instead of guessing. "
    "Answer general questions directly, without calling a tool."
)

# -----------------------------------------------------------------------------
# Confirm the choices
# -----------------------------------------------------------------------------
print(f"Model: {MODEL_ID}")
print(f"Training examples: {NUM_TRAIN_EXAMPLES}, checking examples: {NUM_CHECK_EXAMPLES}")

Model: Qwen/Qwen3-1.7B
Training examples: 1000, checking examples: 100


---
# Part 2 — The three tools

## What we are doing in this part

We write the three tools as plain Python functions, and test them by hand. **No AI is involved yet.** This shows clearly that the tools are just our code: the model's only job, later, is to choose which one to call and with which values.

1. **2.1** Download the real AMFI NAV file and turn it into a list of rows.
2. **2.2** Write `get_nav`: search that list for a fund name.
3. **2.3** Write `get_portfolio`: look up a made-up demo customer.
4. **2.4** Write `start_sip`: check the values and record a made-up SIP.
5. **2.5** Write the **tool descriptions**: the only part the model ever sees.
6. **2.6** Call each tool by hand to see what it returns.

## 2.1 The AMFI NAV file

The file is plain text. Most lines are one scheme each, with 8 values separated by semicolons. A real line from the file:

```
135762;INF846K01WO1;-;Axis Children's Fund;Direct Plan;Growth Option;29.2304;07-Oct-2026
```

| Position | Meaning | Example |
|---|---|---|
| 1 | Scheme code | 135762 |
| 2 | ISIN (Growth / payout) | INF846K01WO1 |
| 3 | ISIN (reinvestment) | `-` (none) |
| 4 | **Scheme name** | Axis Children's Fund |
| 5 | **Plan** | Direct Plan |
| 6 | **Option** | Growth Option |
| 7 | **NAV** | 29.2304 |
| 8 | **Date** | 07-Oct-2026 |

Other lines are headings, such as a fund house name (`Bajaj Finserv Mutual Fund`) or a category (`Open Ended Schemes(Debt Scheme - Gilt Fund)`), or blank. We keep only the lines that start with a scheme code.

One fund appears on several lines: Direct and Regular plans, Growth and IDCW options. Each has its own NAV.

In [5]:
# =============================================================================
# CELL: DOWNLOAD THE AMFI NAV FILE AND TURN IT INTO A LIST OF ROWS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Downloads AMFI's NAV file from the internet (once; we reuse it after).
# 2. Goes through it line by line and keeps only the scheme lines.
# 3. Turns each scheme line into a dictionary with named values:
#        {"scheme_name": ..., "plan": ..., "option": ..., "nav": ..., "date": ...}
# 4. Prints how many scheme rows were found, and shows 3 of them.
#
# IF THE DOWNLOAD FAILS
# ---------------------
# The cell does not crash. It prints a warning and leaves AMFI_ROWS empty.
# Training still works (Part 4 then uses a short built-in list of fund
# names), but get_nav will report that NAV data is not available.
#
# WHAT YOU SHOULD SEE
# -------------------
# "Downloaded ... scheme rows from AMFI" with a number in the thousands,
# followed by 3 example rows.
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION: download_amfi_rows
# -----------------------------------------------------------------------------
# - Input : the web address of the AMFI file
# - Output: a list of dictionaries, one per scheme line
def download_amfi_rows(url):
    """Download the AMFI NAV file and return one dictionary per scheme line."""

    # requests.get(url, timeout=60) downloads the file.
    #   timeout=60 -> give up after 60 seconds instead of waiting forever.
    response = requests.get(url, timeout=60)

    # If the website answered with an error (for example "404 not found"),
    # stop here with that error, instead of trying to read a broken file.
    response.raise_for_status()

    # An empty list that will collect one dictionary per scheme.
    rows = []

    # response.text is the whole file as one piece of text.
    # .splitlines() cuts it into a list of lines.
    for line in response.text.splitlines():

        # Cut the line at every ";" into a list of values.
        parts = line.split(";")

        # Keep only scheme lines:
        #   - they have at least 6 values, and
        #   - the first value is a number (the scheme code).
        # Heading lines and blank lines fail this test and are skipped.
        # ("continue" means: skip the rest of the loop for this line.)
        if len(parts) < 6 or not parts[0].strip().isdigit():
            continue

        # The current file has 8 values per line, with Plan and Option in
        # their own columns. Older versions of the file had 6 values, with
        # no Plan/Option columns. This handles both.
        if len(parts) >= 8:
            scheme_name, plan, option, nav, date = parts[3], parts[4], parts[5], parts[6], parts[7]
        else:
            scheme_name, plan, option, nav, date = parts[3], "", "", parts[4], parts[5]

        # Add one dictionary for this scheme. .strip() removes extra spaces.
        rows.append({
            "scheme_name": scheme_name.strip(),
            "plan": plan.strip(),
            "option": option.strip(),
            "nav": nav.strip(),
            "date": date.strip(),
        })

    # Send the finished list back.
    return rows


# -----------------------------------------------------------------------------
# RUN IT, WITHOUT CRASHING IF THE DOWNLOAD FAILS
# -----------------------------------------------------------------------------
# - "try:" runs the indented code. If anything inside goes wrong (no
#   internet, website down), Python jumps to "except" instead of stopping.
try:
    AMFI_ROWS = download_amfi_rows(AMFI_URL)
    print(f"Downloaded {len(AMFI_ROWS):,} scheme rows from AMFI.")

# - "except Exception as error:" catches any error and stores it in "error".
except Exception as error:
    AMFI_ROWS = []
    print("WARNING: could not download the AMFI file:", error)
    print("Training will still work. get_nav will report that NAV data is not available.")


# -----------------------------------------------------------------------------
# SHOW 3 EXAMPLE ROWS
# -----------------------------------------------------------------------------
# - AMFI_ROWS[:3] -> the first 3 rows (positions 0, 1, 2).
for row in AMFI_ROWS[:3]:
    print(row)

Downloaded 14,357 scheme rows from AMFI.
{'scheme_name': "Axis Children's Fund", 'plan': 'Direct Plan', 'option': 'Growth Option', 'nav': '29.2304', 'date': '07-Oct-2026'}
{'scheme_name': "Axis Children's Fund", 'plan': 'Direct Plan', 'option': 'IDCW Option', 'nav': '26.9265', 'date': '07-Oct-2026'}
{'scheme_name': "Axis Children's Fund", 'plan': 'Regular Plan', 'option': 'Growth Option', 'nav': '25.4417', 'date': '07-Oct-2026'}


**Try it:** add a new cell and run `print(len(AMFI_ROWS))`, or `AMFI_ROWS[500]`, to look at other schemes.

## 2.2 Tool 1: `get_nav`

### What we are doing here

Users won't type a scheme name exactly as AMFI writes it. So `get_nav` uses a simple, loose match: a row matches if **every word** the user gave appears somewhere in the scheme name, ignoring capital letters.

| User's `fund_name` | Matches AMFI scheme name? |
|---|---|
| `Axis Children's Fund` | `Axis Children's Fund` → yes |
| `sbi credit risk` | `SBI CREDIT RISK FUND` → yes (all 3 words found) |
| `Axis Flexi` | `Axis Children's Fund` → no ("Flexi" not found) |

It returns up to 5 matching rows, each with its plan, option, NAV and date. The user can then see every variant (Direct/Regular, Growth/IDCW). We always return the **date** too, because a few schemes in the file have old dates.

In [6]:
# =============================================================================
# CELL: TOOL 1 — get_nav
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Defines the function get_nav(fund_name). Running the cell only DEFINES it;
# we test it in Part 2.6.
#
# HOW IT WORKS
# ------------
# 1. Split the user's fund name into words.
# 2. Keep every AMFI row whose scheme name contains ALL of those words.
# 3. Return up to 5 of them, with plan, option, NAV and date.
#
# WHAT IT RETURNS
# ---------------
# A dictionary. Either:
#     {"fund_name": ..., "total_matches": 4, "matches": [ {...}, {...}, ... ]}
# or, if something is wrong:
#     {"error": "..."}
# Tools return dictionaries because they turn easily into JSON text, which
# we can show to the user or pass back to the model.
# =============================================================================


def get_nav(fund_name):
    """Return the latest NAV rows from AMFI whose scheme name contains every word of fund_name."""

    # If the AMFI file could not be downloaded in 2.1, say so clearly.
    if not AMFI_ROWS:
        return {"error": "AMFI NAV data is not available in this session."}

    # Lower-case the user's text and cut it into words.
    #   "Axis Children's Fund" -> ["axis", "children's", "fund"]
    words = fund_name.lower().split()

    # Go through every AMFI row and keep it only if ALL words appear in its
    # scheme name (also lower-cased).
    #   all(...) is True only when the test is True for every word.
    matches = [row for row in AMFI_ROWS if all(word in row["scheme_name"].lower() for word in words)]

    # No matching scheme: return an error message the user can understand.
    if not matches:
        return {"error": f"No scheme found matching '{fund_name}'."}

    # Return the search text, how many rows matched, and the first 5 rows.
    #   matches[:5] -> at most 5 rows, so the answer stays short.
    return {"fund_name": fund_name, "total_matches": len(matches), "matches": matches[:5]}

## 2.3 Tool 2: `get_portfolio`

### What we are doing here

There is no public data for a real customer's holdings, so we create **5 made-up demo customers**, C1001 to C1005, each holding a few funds. In a real app, this function would query the company's database instead. Everything else in the notebook stays the same.

In [7]:
# =============================================================================
# CELL: TOOL 2 — get_portfolio (with made-up demo customers)
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Creates a small table of 5 made-up customers and the units they hold.
# 2. Defines get_portfolio(customer_id), which looks a customer up.
#
# ALL DATA HERE IS INVENTED for the demo. In a real app, this function would
# read from the company's own database.
# =============================================================================


# -----------------------------------------------------------------------------
# THE DEMO CUSTOMERS
# -----------------------------------------------------------------------------
# - A dictionary: customer ID -> list of holdings.
# - Each holding is a dictionary with a fund name and a number of units.
DEMO_PORTFOLIOS = {
    "C1001": [{"fund": "Parag Parikh Flexi Cap Fund", "units": 152.3}, {"fund": "SBI Small Cap Fund", "units": 88.0}],
    "C1002": [{"fund": "HDFC Flexi Cap Fund", "units": 40.5}],
    "C1003": [{"fund": "Axis Children's Fund", "units": 310.0}, {"fund": "UTI Nifty 50 Index Fund", "units": 95.2}],
    "C1004": [{"fund": "SBI Credit Risk Fund", "units": 1200.0}],
    "C1005": [{"fund": "Mirae Asset Large Cap Fund", "units": 60.0}, {"fund": "Kotak Emerging Equity Fund", "units": 45.7}],
}


def get_portfolio(customer_id):
    """Return the demo holdings of one customer."""

    # .strip() removes spaces; .upper() makes "c1003" the same as "C1003".
    customer_id = customer_id.strip().upper()

    # Unknown customer: return an error message.
    if customer_id not in DEMO_PORTFOLIOS:
        return {"error": f"No customer found with ID {customer_id}. Demo IDs are C1001 to C1005."}

    # Known customer: return the ID and the holdings.
    return {"customer_id": customer_id, "holdings": DEMO_PORTFOLIOS[customer_id]}

## 2.4 Tool 3: `start_sip`

### What we are doing here

`start_sip` checks the values it receives, then **records** the SIP in a list and returns a confirmation ID. **Nothing real happens**: no money moves. In a real app this function would call the platform's order system.

It also shows something important: **tools should check their own inputs.** Even a well-trained model can make mistakes, so the function rejects an amount that is not a whole number, or a day outside 1 to 28.

In [8]:
# =============================================================================
# CELL: TOOL 3 — start_sip (records a made-up SIP)
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Creates an empty list, SIP_LEDGER, to record every SIP started.
# 2. Defines start_sip(fund_name, amount, day_of_month), which:
#      - checks that amount is a whole number of at least 100
#      - checks that day_of_month is a whole number from 1 to 28
#      - records the SIP and returns a confirmation ID such as "SIP-0001"
#
# NOTHING REAL HAPPENS. This is a demo. A real app would place an order here.
#
# WHY DAYS 1 TO 28?
# -----------------
# Every month has at least 28 days, so these dates always exist. Many SIP
# platforms use the same rule.
# =============================================================================


# An empty list. Each started SIP is added here.
SIP_LEDGER = []


def start_sip(fund_name, amount, day_of_month):
    """Record a demo monthly SIP after checking its values."""

    # isinstance(amount, int) -> is amount a whole number?
    # (A model mistake such as amount="5000 rupees" is text, not a number,
    #  and is rejected here.)
    if not isinstance(amount, int) or amount < 100:
        return {"error": "amount must be a whole number of rupees, at least 100."}

    # The same check for the day.
    if not isinstance(day_of_month, int) or not 1 <= day_of_month <= 28:
        return {"error": "day_of_month must be a whole number from 1 to 28."}

    # Make a confirmation ID from the count of SIPs so far:
    #   first SIP -> "SIP-0001". {...:04d} pads the number to 4 digits.
    sip_id = f"SIP-{len(SIP_LEDGER) + 1:04d}"

    # Write down the SIP as a dictionary.
    sip = {"sip_id": sip_id, "fund_name": fund_name, "amount": amount, "day_of_month": day_of_month}

    # Add it to the ledger.
    SIP_LEDGER.append(sip)

    # Return the status and the details.
    return {"status": "created (demo only, no real order placed)", **sip}

## 2.5 The tool descriptions: what the model actually sees

### What we are doing here

The model never sees the Python code above. It only sees a **description** of each tool, written in a standard JSON format called a *JSON schema*:

- `name`: the exact name the model must write.
- `description`: what the tool does, in plain English. The model uses this to decide **when** to use it.
- `parameters`: the values it needs, with the **type** of each one (`string` = text, `integer` = whole number), and which ones are `required`.

Writing clear descriptions matters: the model's decision is only as good as what these descriptions tell it.

We also build a small lookup table, `TOOL_FUNCTIONS`, that connects each tool **name** (text) to the real Python **function**. This is how our code turns the model's decision into an actual function call.

In [9]:
# =============================================================================
# CELL: THE TOOL DESCRIPTIONS (WHAT THE MODEL SEES) AND THE NAME -> FUNCTION TABLE
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. TOOLS: a list of 3 tool descriptions in the standard JSON-schema format.
#    These are given to the model with every request, in training and in use.
# 2. TOOL_FUNCTIONS: connects each tool name to the real Python function.
# 3. run_tool(name, arguments): runs the right function safely.
#
# THE FORMAT OF ONE DESCRIPTION
# -----------------------------
#   {"type": "function",
#    "function": {"name": ...,
#                 "description": ...,
#                 "parameters": {"type": "object",
#                                "properties": { value_name: {"type": ..., "description": ...} },
#                                "required": [ ...names of values that must be given... ]}}}
# =============================================================================


TOOLS = [

    # -------------------------------------------------------------------------
    # Description of tool 1: get_nav
    # -------------------------------------------------------------------------
    {
        "type": "function",
        "function": {
            "name": "get_nav",
            "description": "Get the latest NAV (price of one unit) of a mutual fund scheme from AMFI.",
            "parameters": {
                "type": "object",
                "properties": {
                    # "string" = text. The model should copy the name as the user wrote it.
                    "fund_name": {"type": "string", "description": "Name of the mutual fund scheme, as written by the user."},
                },
                "required": ["fund_name"],
            },
        },
    },

    # -------------------------------------------------------------------------
    # Description of tool 2: get_portfolio
    # -------------------------------------------------------------------------
    {
        "type": "function",
        "function": {
            "name": "get_portfolio",
            "description": "Get the mutual fund holdings of a customer.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string", "description": "The customer's ID, for example C1001."},
                },
                "required": ["customer_id"],
            },
        },
    },

    # -------------------------------------------------------------------------
    # Description of tool 3: start_sip
    # -------------------------------------------------------------------------
    {
        "type": "function",
        "function": {
            "name": "start_sip",
            "description": "Start a monthly SIP (Systematic Investment Plan) in a mutual fund.",
            "parameters": {
                "type": "object",
                "properties": {
                    "fund_name": {"type": "string", "description": "Name of the mutual fund scheme, as written by the user."},
                    # "integer" = a whole number. This is why "5000 rupees" counts as wrong.
                    "amount": {"type": "integer", "description": "Monthly amount in rupees, as a whole number, for example 5000."},
                    "day_of_month": {"type": "integer", "description": "Day of the month for the debit, from 1 to 28."},
                },
                "required": ["fund_name", "amount", "day_of_month"],
            },
        },
    },
]


# -----------------------------------------------------------------------------
# THE NAME -> FUNCTION TABLE
# -----------------------------------------------------------------------------
# - The model writes a tool NAME as text, for example "get_nav".
# - This dictionary finds the real Python function for that text.
#   (Notice: no brackets after the function names. We store the functions
#    themselves, without running them.)
TOOL_FUNCTIONS = {"get_nav": get_nav, "get_portfolio": get_portfolio, "start_sip": start_sip}


# -----------------------------------------------------------------------------
# FUNCTION: run_tool
# -----------------------------------------------------------------------------
# - Input : a tool name and a dictionary of argument values
# - Output: whatever the tool returns, or an error dictionary
# - It never crashes, even if the model asks for a tool that does not exist
#   or gives the wrong values.
def run_tool(name, arguments):
    """Run the Python function behind a tool name, safely."""

    # Look the name up. .get() gives None if the name is not in the table.
    function = TOOL_FUNCTIONS.get(name)

    # Unknown tool name (for example, the model invented one).
    if function is None:
        return {"error": f"Unknown tool: {name}"}

    try:
        # **arguments unpacks the dictionary into named values:
        #   run_tool("get_nav", {"fund_name": "X"})  ->  get_nav(fund_name="X")
        return function(**arguments)

    # TypeError happens when the values do not fit the function, for example
    # a missing or extra argument name.
    except TypeError as error:
        return {"error": f"Wrong arguments for {name}: {error}"}


# Confirm how many tool descriptions we wrote.
print(f"{len(TOOLS)} tools described: {[tool['function']['name'] for tool in TOOLS]}")

3 tools described: ['get_nav', 'get_portfolio', 'start_sip']


## 2.6 Test the tools by hand

### What we are doing here

We call each tool directly, the same way our code will call it after the model decides. This is the **"our code does"** half of the system, working on its own.

**Predict first:** what will `start_sip` return when the amount is the text `"5000 rupees"`?

In [10]:
# =============================================================================
# CELL: CALL EACH TOOL BY HAND
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Runs each tool through run_tool(), exactly as our code will do later when
# the model chooses a tool. Prints each result as neat JSON.
#
# The last test passes a WRONG value on purpose ("5000 rupees" as text) to
# show that start_sip rejects it.
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION: show
# -----------------------------------------------------------------------------
# - A small helper that prints a result as indented JSON, so it is easy to
#   read.
#   indent=2           -> indent nested parts by 2 spaces
#   ensure_ascii=False -> show characters such as ₹ as they are
def show(result):
    print(json.dumps(result, indent=2, ensure_ascii=False))


# Test 1: a real NAV lookup from the AMFI data.
print("get_nav('Axis Children's Fund'):")
show(run_tool("get_nav", {"fund_name": "Axis Children's Fund"}))

# Test 2: a demo customer's holdings.
print("\nget_portfolio('C1003'):")
show(run_tool("get_portfolio", {"customer_id": "C1003"}))

# Test 3: a valid SIP. The amount and day are whole numbers.
print("\nstart_sip('Axis Children's Fund', 5000, 10):")
show(run_tool("start_sip", {"fund_name": "Axis Children's Fund", "amount": 5000, "day_of_month": 10}))

# Test 4: an INVALID SIP, with the amount as text. Expect an error.
print("\nstart_sip with amount = '5000 rupees' (wrong on purpose):")
show(run_tool("start_sip", {"fund_name": "Axis Children's Fund", "amount": "5000 rupees", "day_of_month": 10}))

get_nav('Axis Children's Fund'):
{
  "fund_name": "Axis Children's Fund",
  "total_matches": 8,
  "matches": [
    {
      "scheme_name": "Axis Children's Fund",
      "plan": "Direct Plan",
      "option": "Growth Option",
      "nav": "29.2304",
      "date": "07-Oct-2026"
    },
    {
      "scheme_name": "Axis Children's Fund",
      "plan": "Direct Plan",
      "option": "IDCW Option",
      "nav": "26.9265",
      "date": "07-Oct-2026"
    },
    {
      "scheme_name": "Axis Children's Fund",
      "plan": "Regular Plan",
      "option": "Growth Option",
      "nav": "25.4417",
      "date": "07-Oct-2026"
    },
    {
      "scheme_name": "Axis Children's Fund",
      "plan": "Regular Plan",
      "option": "IDCW Option",
      "nav": "23.4720",
      "date": "07-Oct-2026"
    },
    {
      "scheme_name": "Axis Children's Fund",
      "plan": "Direct Plan",
      "option": "Growth Option",
      "nav": "29.7894",
      "date": "07-Oct-2026"
    }
  ]
}

get_portfolio('C1003'):
{

**Try it:** change `"Axis Children's Fund"` in the first test to another fund, such as `"sbi small cap"` or `"parag parikh flexi"`.

<details>
<summary><b>Check yourself:</b> where does the model fit into what we just did? (click to open)</summary>

It doesn't yet. Everything in Part 2 is ordinary Python. Later, the model's only job is to produce the first argument of `run_tool` (the tool name) and the second (the values), from a sentence typed by a user.
</details>

---
# Part 3 — Load Qwen3

## What we are doing in this part

1. **3.1** Load Qwen3's **tokenizer** (text ↔ token IDs) and look at its special tokens.
2. **3.2** Build the **exact prompt** the model receives for one request: system instruction + the three tool descriptions + the user's request. Seeing this makes the whole idea concrete.
3. **3.3** Download Qwen3-1.7B and load it in **4-bit** form so it fits easily on the T4.

**What goes in:** the model name `Qwen/Qwen3-1.7B`.
**What comes out:** `tokenizer` and `model`, ready on the GPU.

## 3.1 The tokenizer and its special tokens

Qwen3 marks the structure of a conversation with special tokens:

| Token | Meaning |
|---|---|
| `<\|im_start\|>` | "a new message starts here", followed by `system`, `user` or `assistant` |
| `<\|im_end\|>` | "this message is finished": the **stop token**. The model writes it when its answer is complete |
| `<\|endoftext\|>` | used as the **padding token** (filler for short examples in a batch) |
| `<tool_call>` … `</tool_call>` | wrap a tool call written by the model |
| `<think>` … `</think>` | wrap Qwen3's "thinking" text. We switch thinking off, so this stays empty |

**Good news compared with TinyLlama:** in TinyLlama the stop token and the padding token were the same, and we had to fix it. In Qwen3 they are **different tokens**, so no fix is needed. The next cell confirms this.

In [11]:
# =============================================================================
# CELL: LOAD THE TOKENIZER AND CHECK ITS SPECIAL TOKENS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Downloads Qwen3's tokenizer from huggingface.co.
# 2. Prints the stop token and the padding token, and checks they differ.
# 3. Splits one sample request into tokens, so we can see how text becomes
#    numbers.
#
# WHAT YOU SHOULD SEE
# -------------------
#     Stop token    : <|im_end|>      (ID 151645)
#     Padding token : <|endoftext|>   (ID 151643)
#     They are different, so the stop token will be learned.
# followed by the sample request as token IDs and pieces.
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: load the tokenizer
# -----------------------------------------------------------------------------
# - AutoTokenizer.from_pretrained(MODEL_ID) downloads the tokenizer files that
#   belong to Qwen3-1.7B and loads them.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# -----------------------------------------------------------------------------
# LINE 2: pad short examples on the right
# -----------------------------------------------------------------------------
# - When examples in a batch have different lengths, filler (padding) is
#   added at the END of the shorter ones.
tokenizer.padding_side = "right"


# -----------------------------------------------------------------------------
# LINES 3-4: show the stop token and the padding token
# -----------------------------------------------------------------------------
# - eos_token = "end of sequence" token = the stop token.
print(f"Stop token    : {tokenizer.eos_token}   (ID {tokenizer.eos_token_id})")
print(f"Padding token : {tokenizer.pad_token}   (ID {tokenizer.pad_token_id})")

# -----------------------------------------------------------------------------
# LINE 5: check they are different
# -----------------------------------------------------------------------------
# - assert stops the notebook with the message if the condition is False.
assert tokenizer.eos_token_id != tokenizer.pad_token_id, "Stop and padding tokens are the same: padding would hide the stop token."
print("They are different, so the stop token will be learned.\n")


# -----------------------------------------------------------------------------
# LINES 6-9: see one request as tokens
# -----------------------------------------------------------------------------
# - tokenizer(text)["input_ids"] -> the list of token ID numbers.
# - convert_ids_to_tokens(...)   -> the text piece behind each ID.
#   "Ġ" in a piece means "there was a space before this piece".
sample_request = "Start a SIP of ₹5,000 in Axis Children's Fund on the 10th"
sample_ids = tokenizer(sample_request)["input_ids"]
print("Request:", sample_request)
print("IDs    :", sample_ids)
print("Pieces :", tokenizer.convert_ids_to_tokens(sample_ids))
print("Count  :", len(sample_ids), "tokens")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Stop token    : <|im_end|>   (ID 151645)
Padding token : <|endoftext|>   (ID 151643)
They are different, so the stop token will be learned.

Request: Start a SIP of ₹5,000 in Axis Children's Fund on the 10th
IDs    : [3479, 264, 65441, 315, 89791, 20, 11, 15, 15, 15, 304, 34474, 15044, 594, 13190, 389, 279, 220, 16, 15, 339]
Pieces : ['Start', 'Ġa', 'ĠSIP', 'Ġof', 'ĠâĤ¹', '5', ',', '0', '0', '0', 'Ġin', 'ĠAxis', 'ĠChildren', "'s", 'ĠFund', 'Ġon', 'Ġthe', 'Ġ', '1', '0', 'th']
Count  : 21 tokens


## 3.2 The exact prompt the model receives

### What we are doing here

For every request, the tokenizer's **chat template** builds one long text containing:

1. a **system** message: our `SYSTEM_PROMPT`, then a fixed block that lists the three tool descriptions inside `<tools> … </tools>`, and explains the `<tool_call>` format;
2. the **user** message: the request;
3. the start of the **assistant** message, with an empty `<think></think>` block because thinking is switched off. The model continues writing from here.

We never type this by hand. `apply_chat_template(messages, tools=TOOLS, ...)` builds it.

**Predict first:** the user's request is about 20 tokens. How long do you think the whole prompt will be? (Most of it is the tool descriptions.)

In [12]:
# =============================================================================
# CELL: SEE THE FULL PROMPT FOR ONE REQUEST
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Writes one conversation: our system instruction + one user request.
# 2. Asks the tokenizer to build Qwen3's full prompt, including the three
#    tool descriptions.
# 3. Prints the prompt and its length in tokens.
#
# THE TWO IMPORTANT SWITCHES
# --------------------------
#   tools=TOOLS             -> include our 3 tool descriptions in the prompt
#   enable_thinking=False   -> switch off Qwen3's "thinking" step, so it
#                              answers directly. The prompt then ends with an
#                              empty "<think>\n\n</think>" block.
# =============================================================================


# -----------------------------------------------------------------------------
# PART 1: the conversation as a Python list
# -----------------------------------------------------------------------------
# - Each message is a dictionary with a "role" (who speaks) and "content".
demo_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the NAV of Axis Children's Fund?"},
]

# -----------------------------------------------------------------------------
# PART 2: build the full prompt text
# -----------------------------------------------------------------------------
demo_prompt = tokenizer.apply_chat_template(
    demo_messages,               # the conversation above
    tools=TOOLS,                 # add the 3 tool descriptions from Part 2.5
    tokenize=False,              # give back readable text, not token IDs
    add_generation_prompt=True,  # end with the start of the assistant's turn
    enable_thinking=False,       # Qwen3-specific: no thinking step
)

# -----------------------------------------------------------------------------
# PART 3: show it
# -----------------------------------------------------------------------------
print(demo_prompt)
print("-" * 80)

# Count its tokens. add_special_tokens=False: count exactly this text, adding nothing.
print("Prompt length:", len(tokenizer(demo_prompt, add_special_tokens=False)["input_ids"]), "tokens")

<|im_start|>system
You are a mutual fund assistant for an Indian investment app. Call a tool when the user needs a fund's NAV, their portfolio, or wants to start a SIP. If a detail the tool needs is missing, ask the user for it instead of guessing. Answer general questions directly, without calling a tool.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_nav", "description": "Get the latest NAV (price of one unit) of a mutual fund scheme from AMFI.", "parameters": {"type": "object", "properties": {"fund_name": {"type": "string", "description": "Name of the mutual fund scheme, as written by the user."}}, "required": ["fund_name"]}}}
{"type": "function", "function": {"name": "get_portfolio", "description": "Get the mutual fund holdings of a customer.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string",

Read the printed prompt from top to bottom:

- `<|im_start|>system` … our instruction, then `# Tools`, then the three descriptions inside `<tools>…</tools>`, then the instruction to answer with `<tool_call>{"name": …, "arguments": …}</tool_call>`.
- `<|im_start|>user` … the request.
- `<|im_start|>assistant` followed by the empty `<think></think>` block: the model writes its answer from here.

So the model is **told** the tools in plain text, every time. Fine-tuning teaches it to use that information well.

## 3.3 Load the model in 4-bit

### What we are doing here

We download Qwen3-1.7B (about 3.4 GB) and compress it to 4 bits while loading, exactly as in the TinyLlama notebook. Then we count its parameters and measure its memory.

By my calculation from the model's published sizes:

- **Parameters:** about **1.72 billion**.
- **32-bit memory:** about **6.4 GB**.
- **4-bit memory:** roughly **1.3 GB**. Most numbers become 4-bit; the word table stays 16-bit.

The cell prints the real figures.

In [13]:
# =============================================================================
# CELL: LOAD QWEN3-1.7B IN 4-BIT
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Writes down the 4-bit compression settings.
# 2. Downloads Qwen3-1.7B and loads it onto the GPU, compressed to 4 bits.
# 3. Counts its parameters and measures its GPU memory.
#
# WHY 4-BIT
# ---------
# 1.72 billion numbers x 4 bytes (32-bit) = about 6.4 GB just to hold the
# model, before any training memory. In 4 bits, each number takes 1/8 of
# that space.
#
# ABOUT A WARNING YOU MAY SEE
# ---------------------------
# "You are sending unauthenticated requests to the HF Hub" only means we are
# not logged in to Hugging Face. Downloads still work.
# =============================================================================


# -----------------------------------------------------------------------------
# PART 1: the compression settings
# -----------------------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # store each parameter in 4 bits instead of 32
    bnb_4bit_quant_type="nf4",             # the 16 allowed levels sit closer together near zero, where most weights are
    bnb_4bit_use_double_quant=True,        # also compress the scale numbers (one per group of 64 weights)
    bnb_4bit_compute_dtype=torch.float16,  # expand to 16-bit only while calculating; the T4 is fast at 16-bit
)


# -----------------------------------------------------------------------------
# PART 2: download and load the model
# -----------------------------------------------------------------------------
# - AutoModelForCausalLM: a model that predicts the next token ("causal LM").
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,                        # "Qwen/Qwen3-1.7B"
    quantization_config=bnb_config,  # compress to 4 bits while loading
    device_map="auto",               # put it on the GPU
)


# -----------------------------------------------------------------------------
# PART 3: count the parameters
# -----------------------------------------------------------------------------
# - In a 4-bit block ("Params4bit"), two numbers are packed into each byte,
#   so numel() counts only half of them. We double those to get the real
#   count. (Without this, the count comes out far too low.)
total_params = 0
for p in model.parameters():                       # every block of numbers in the model
    if p.__class__.__name__ == "Params4bit":       # a 4-bit compressed block?
        total_params += p.numel() * 2              # yes: double the stored count
    else:
        total_params += p.numel()                  # no: the stored count is right


# -----------------------------------------------------------------------------
# PART 4: measure the memory and show the results
# -----------------------------------------------------------------------------
# - get_memory_footprint() -> bytes used by the model now; / 1024**3 -> GB.
memory_gb = model.get_memory_footprint() / 1024**3

print(f"Parameters in the model : {total_params:,}")                        # about 1.72 billion
print(f"Memory if stored 32-bit : {total_params * 4 / 1024**3:.2f} GB")    # about 6.4 GB
print(f"Memory used right now   : {memory_gb:.2f} GB")                     # roughly 1.3 GB

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Parameters in the model : 1,720,574,976
Memory if stored 32-bit : 6.41 GB
Memory used right now   : 1.24 GB


---
# Part 4 — Generate the training data

## What we are doing in this part

We create every training and checking example ourselves, with plain Python. No other AI model is used, so the data is free, repeatable, and fully visible.

1. **4.1 Fund names.** Take real scheme names from the AMFI file. Keep 80% for training and **20% only for checking**.
2. **4.2 Sentence patterns.** Write several ways of asking for each tool, in English and some Hinglish. Again, some patterns are kept **only for checking**.
3. **4.3 The generator.** Pick a pattern and fill it with a fund name, an amount, a day or a customer ID. Because we chose those values, we also know the correct answer, and write it automatically.
4. **4.4 Build the two sets.** 1,000 training examples and 100 checking examples, then look at some.

**What goes in:** AMFI scheme names, our sentence patterns, random values.
**What comes out:** `train_examples` and `check_examples`, two Python lists.

## How one example is made

```
Pattern:          "Start a SIP of {amount} in {fund} on the {day} of every month."
Random values:    amount = 7500   → written as "Rs 7500"
                  fund   = "SBI Credit Risk Fund"
                  day    = 5      → written as "5th"

User request:     "Start a SIP of Rs 7500 in SBI Credit Risk Fund on the 5th of every month."
Correct answer:   start_sip  with  {"fund_name": "SBI Credit Risk Fund", "amount": 7500, "day_of_month": 5}
```

Notice that the **request** says "Rs 7500" and "5th", but the **correct answer** has the whole numbers `7500` and `5`. That conversion is exactly what we want the model to learn.

## Why some patterns and fund names are kept for checking only

If the checking set used the same sentences and fund names as training, the model could score well just by recognising sentences it had memorised. By checking on **new wording and new fund names**, the score tells us whether it really learned the skill.

In [14]:
# =============================================================================
# CELL: 4.1 — FUND NAMES FOR THE EXAMPLES (REAL NAMES FROM AMFI)
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Takes the scheme names from the AMFI rows downloaded in Part 2.1.
# 2. Cleans them (removes plan/option text after " - ", drops very long or
#    very short names) and removes duplicates.
# 3. If AMFI could not be downloaded, uses a short built-in list instead.
# 4. Shuffles the names and splits them:
#        80% -> used only in TRAINING examples
#        20% -> used only in CHECKING examples
#
# WHAT YOU SHOULD SEE
# -------------------
# How many unique names were found, how many went to each side, and a few
# examples from each.
# =============================================================================


# -----------------------------------------------------------------------------
# A random-number generator with a fixed seed
# -----------------------------------------------------------------------------
# - random.Random(SEED) makes our own generator. With the same seed, it makes
#   the same "random" choices every run, so everyone gets the same data.
rng = random.Random(SEED)


# -----------------------------------------------------------------------------
# A short backup list, used only if the AMFI download failed
# -----------------------------------------------------------------------------
FALLBACK_FUND_NAMES = [
    "Axis Children's Fund", "Franklin India Banking & PSU Debt Fund", "ITI Banking & PSU Debt Fund",
    "SBI Credit Risk Fund", "Parag Parikh Flexi Cap Fund", "HDFC Flexi Cap Fund", "SBI Small Cap Fund",
    "Mirae Asset Large Cap Fund", "Kotak Emerging Equity Fund", "Nippon India Small Cap Fund",
    "UTI Nifty 50 Index Fund", "ICICI Prudential Bluechip Fund", "DSP Midcap Fund", "Axis Midcap Fund",
    "Canara Robeco Bluechip Equity Fund", "HDFC Mid-Cap Opportunities Fund", "SBI Equity Hybrid Fund",
    "Quant Small Cap Fund", "Kotak Flexicap Fund", "Tata Digital India Fund",
]


# -----------------------------------------------------------------------------
# FUNCTION: clean_fund_name
# -----------------------------------------------------------------------------
# - Some AMFI names include the plan and option after " - ", for example
#   "XYZ Equity Fund - Direct Plan - Growth". We keep only the part before the
#   first " - ", which is the fund's own name.
# - " ".join(name.split()) turns any run of spaces into a single space.
def clean_fund_name(name):
    name = name.split(" - ")[0]
    return " ".join(name.split())


# -----------------------------------------------------------------------------
# Collect unique, cleaned names from AMFI
# -----------------------------------------------------------------------------
fund_names = []          # the names we keep
seen = set()             # lower-cased names already kept, to skip duplicates
for row in AMFI_ROWS:
    name = clean_fund_name(row["scheme_name"])
    # Skip names that are very short or very long (long ones are usually
    # closed-ended series with codes and day counts).
    if not 8 <= len(name) <= 50:
        continue
    # .casefold() is a stronger lower-case, so "SBI X" and "sbi x" count as the same.
    if name.casefold() in seen:
        continue
    seen.add(name.casefold())
    fund_names.append(name)


# -----------------------------------------------------------------------------
# Use the backup list if AMFI gave us too few names
# -----------------------------------------------------------------------------
if len(fund_names) < 20:
    print("Using the built-in backup list of fund names.")
    fund_names = list(FALLBACK_FUND_NAMES)


# -----------------------------------------------------------------------------
# Shuffle, then split 80% / 20%
# -----------------------------------------------------------------------------
# - sorted() first, so the order before shuffling is the same on every run.
fund_names = sorted(fund_names)
rng.shuffle(fund_names)

# Number of names for checking: 20%, but at least 4.
check_name_count = max(4, len(fund_names) // 5)

# The first 20% go to checking; the rest go to training.
CHECK_FUND_NAMES = fund_names[:check_name_count]
TRAIN_FUND_NAMES = fund_names[check_name_count:]


# -----------------------------------------------------------------------------
# Show the result
# -----------------------------------------------------------------------------
print(f"Unique fund names: {len(fund_names):,}")
print(f"  for training: {len(TRAIN_FUND_NAMES):,}   e.g. {TRAIN_FUND_NAMES[:3]}")
print(f"  for checking: {len(CHECK_FUND_NAMES):,}   e.g. {CHECK_FUND_NAMES[:3]}")

Unique fund names: 2,360
  for training: 1,888   e.g. ['UTI Small Cap Fund', 'BANDHAN CRISIL IBX GILT JUNE 2027 INDEX FUND', 'Axis ELSS- Tax Saver Fund']
  for checking: 472   e.g. ['Mirae Asset Ultra Short to Short Term Fund', 'Navi ELSS Tax Saver Fund', 'Mirae Asset Nifty LargeMidcap 250 Index Fund']


## 4.2 The sentence patterns

### What we are doing here

For each kind of request, we write several sentence patterns. Words in `{curly brackets}` are filled in later:

| Placeholder | Filled with | Example |
|---|---|---|
| `{fund}` | a real fund name, sometimes lower-cased | `SBI Credit Risk Fund` |
| `{amount}` | a monthly amount, written in one of several styles | `₹5,000`, `Rs 5000`, `5000 rupees`, `INR 5,000`, `5k`, `5000` |
| `{day}` | a day of the month as an ordinal | `10th` |
| `{day_num}` | the same day as a plain number | `10` |
| `{cid}` | a customer ID | `C4821` |

The **missing-details** patterns come in pairs: the user's request, and the question the assistant should ask back.

The **general questions** are fixed question-and-answer pairs. These answers are short, general explanations written for this demo. They are not financial advice.

Each kind has a `TRAIN` list and a separate `CHECK` list, with different wording.

In [15]:
# =============================================================================
# CELL: 4.2 — THE SENTENCE PATTERNS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Defines all sentence patterns as Python lists. Nothing is generated yet.
#
#   NAV_TRAIN / NAV_CHECK               -> requests that need get_nav
#   PORTFOLIO_TRAIN / PORTFOLIO_CHECK   -> requests that need get_portfolio
#   SIP_TRAIN / SIP_CHECK               -> requests that need start_sip
#   MISSING_TRAIN / MISSING_CHECK       -> (request, question to ask back):
#                                          a needed detail is missing
#   GENERAL_TRAIN / GENERAL_CHECK       -> (question, answer): no tool needed
#
# The CHECK lists use wording that never appears in the TRAIN lists.
# =============================================================================


# -----------------------------------------------------------------------------
# Requests that need get_nav
# -----------------------------------------------------------------------------
NAV_TRAIN = [
    "What is the NAV of {fund}?",
    "What's the current NAV of {fund}?",
    "{fund} NAV?",
    "Tell me today's NAV for {fund}.",
    "How much is one unit of {fund} worth right now?",
    "Can you check the latest NAV of {fund}?",
    "I want to know the NAV of {fund}.",
    "NAV for {fund} please",
    "{fund} ka NAV kya hai?",
    "Please fetch the NAV of {fund}.",
]
NAV_CHECK = [
    "Could you look up the net asset value of {fund}?",
    "{fund} ka aaj ka NAV batao",
    "what's {fund} trading at today, nav wise?",
]

# -----------------------------------------------------------------------------
# Requests that need get_portfolio
# -----------------------------------------------------------------------------
PORTFOLIO_TRAIN = [
    "Show my portfolio. My customer ID is {cid}.",
    "What funds do I hold? Customer ID {cid}.",
    "Show holdings for customer {cid}.",
    "Can you show me the portfolio of {cid}?",
    "I am customer {cid}, what are my investments?",
    "Mera portfolio dikhao, customer ID {cid}.",
    "List all holdings for {cid}",
]
PORTFOLIO_CHECK = [
    "Customer {cid} here. Please pull up my mutual fund holdings.",
    "{cid} ka portfolio batao",
]

# -----------------------------------------------------------------------------
# Requests that need start_sip
# -----------------------------------------------------------------------------
SIP_TRAIN = [
    "Start a SIP of {amount} in {fund} on the {day} of every month.",
    "I want to invest {amount} every month in {fund} on the {day}.",
    "Set up a monthly SIP in {fund} for {amount}, date {day_num}.",
    "Please start a {amount} SIP in {fund}, debit on the {day}.",
    "Begin an SIP: {fund}, {amount} per month, on the {day}.",
    "{fund} mein {amount} ki SIP shuru karo, har mahine {day_num} tareekh ko.",
    "Can you create a SIP of {amount} in {fund}? Date: {day_num}.",
]
SIP_CHECK = [
    "Register a monthly investment of {amount} into {fund}, to be debited on the {day}.",
    "{amount} har mahine {fund} mein daalna hai, {day} ko.",
]

# -----------------------------------------------------------------------------
# A needed detail is missing: (user request, question the assistant should ask)
# -----------------------------------------------------------------------------
MISSING_TRAIN = [
    ("Start a SIP in {fund} on the {day} of every month.", "How much would you like to invest every month in {fund}?"),
    ("Start a SIP of {amount} in {fund}.", "On which day of the month (1 to 28) should the SIP of {amount_clean} in {fund} be debited?"),
    ("I want to start a SIP in {fund}.", "Sure. How much would you like to invest every month, and on which day of the month (1 to 28) should it be debited?"),
    ("Show my portfolio.", "Could you share your customer ID so I can fetch your portfolio?"),
    ("What funds do I currently hold?", "Could you share your customer ID so I can fetch your portfolio?"),
    ("What is the NAV?", "Which fund's NAV would you like me to check?"),
    ("Mujhe SIP start karni hai.", "Sure. Which fund would you like to invest in, how much every month, and on which day of the month (1 to 28)?"),
]
MISSING_CHECK = [
    ("Please set up a monthly SIP for me in {fund}.", "Sure. How much would you like to invest every month, and on which day of the month (1 to 28) should it be debited?"),
    ("Mera portfolio dikhao", "Could you share your customer ID so I can fetch your portfolio?"),
    ("Check the latest NAV for me.", "Which fund's NAV would you like me to check?"),
]

# -----------------------------------------------------------------------------
# General questions: (question, answer). No tool is needed.
# These answers are short demo explanations, not financial advice.
# -----------------------------------------------------------------------------
GENERAL_TRAIN = [
    ("What is an SIP?", "An SIP (Systematic Investment Plan) invests a fixed amount in a mutual fund at regular intervals, usually every month."),
    ("What does NAV mean?", "NAV (Net Asset Value) is the price of one unit of a mutual fund. It is calculated at the end of every business day."),
    ("What is the difference between a direct plan and a regular plan?", "A direct plan is bought straight from the fund house and has a lower expense ratio. A regular plan is bought through a distributor, who is paid from the fund's expenses."),
    ("What is an expense ratio?", "The expense ratio is the yearly fee a mutual fund charges, shown as a percentage of your investment."),
    ("What is the difference between Growth and IDCW options?", "In the Growth option, profits stay invested and the NAV grows. In the IDCW option, the fund pays out part of the profits to you from time to time."),
    ("What is an ELSS fund?", "An ELSS (Equity Linked Savings Scheme) is an equity mutual fund with a 3-year lock-in. Under the old tax regime it qualifies for a deduction under Section 80C."),
    ("What is an exit load?", "An exit load is a fee some funds charge if you redeem your units before a set period, for example 1% within one year."),
    ("What is a debt fund?", "A debt fund invests mainly in bonds, such as government securities and corporate bonds."),
    ("What is an index fund?", "An index fund copies a market index, such as the Nifty 50, instead of having a manager pick stocks. Its costs are usually low."),
    ("Can I stop my SIP anytime?", "Yes. Most SIPs can be stopped at any time without a penalty. Units already bought stay invested."),
    ("What is a liquid fund?", "A liquid fund invests in very short-term debt, with maturities of up to 91 days. It is often used to park money for a short time."),
    ("What is AUM?", "AUM (Assets Under Management) is the total market value of all the money a mutual fund manages."),
    ("What is a large cap fund?", "A large cap fund invests mainly in the 100 largest listed companies by market value."),
    ("Hello", "Hello! I can check fund NAVs, show your portfolio, or start a SIP. How can I help?"),
    ("Thank you", "You're welcome! Let me know if you need anything else."),
    ("What can you do?", "I can check the latest NAV of a mutual fund, show your portfolio if you share your customer ID, and start a monthly SIP."),
    ("SIP kya hota hai?", "SIP (Systematic Investment Plan) mein aap har mahine ek fixed amount mutual fund mein invest karte ho."),
]
GENERAL_CHECK = [
    ("What is a small cap fund?", "A small cap fund invests mainly in smaller companies, ranked 251st and below by market value. It can grow faster but is riskier."),
    ("Is a mutual fund safe?", "Mutual funds are regulated by SEBI, but their value goes up and down with the market. The risk depends on the type of fund."),
    ("What is a lump sum investment?", "A lump sum investment means investing a large amount at once, instead of in monthly instalments like an SIP."),
    ("Good morning", "Good morning! I can check NAVs, show your portfolio, or start a SIP for you."),
    ("NAV roz badalta hai kya?", "Haan, NAV har business day ke end mein calculate hota hai, isliye roz badal sakta hai."),
]

# -----------------------------------------------------------------------------
# Confirm how many patterns we have
# -----------------------------------------------------------------------------
print(f"get_nav       : {len(NAV_TRAIN)} training patterns, {len(NAV_CHECK)} checking patterns")
print(f"get_portfolio : {len(PORTFOLIO_TRAIN)} training patterns, {len(PORTFOLIO_CHECK)} checking patterns")
print(f"start_sip     : {len(SIP_TRAIN)} training patterns, {len(SIP_CHECK)} checking patterns")
print(f"missing detail: {len(MISSING_TRAIN)} training patterns, {len(MISSING_CHECK)} checking patterns")
print(f"general       : {len(GENERAL_TRAIN)} training questions, {len(GENERAL_CHECK)} checking questions")

get_nav       : 10 training patterns, 3 checking patterns
get_portfolio : 7 training patterns, 2 checking patterns
start_sip     : 7 training patterns, 2 checking patterns
missing detail: 7 training patterns, 3 checking patterns
general       : 17 training questions, 5 checking questions


## 4.3 The generator

### What we are doing here

We write small functions that turn **one pattern + random values** into **one example with its correct answer**. Every example is a dictionary like this:

| Key | Meaning | Example |
|---|---|---|
| `kind` | which of the 5 kinds it is | `"start_sip"` |
| `user` | the request text | `"Start a SIP of Rs 7500 in SBI Credit Risk Fund on the 5th of every month."` |
| `tool` | the correct tool name, or `None` if no tool should be called | `"start_sip"` |
| `arguments` | the correct values, or `None` | `{"fund_name": "SBI Credit Risk Fund", "amount": 7500, "day_of_month": 5}` |
| `reply` | for no-tool kinds: the correct text answer, otherwise `None` | `None` |

In [16]:
# =============================================================================
# CELL: 4.3 — THE EXAMPLE GENERATOR
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Defines the functions that create examples. Nothing is generated yet; the
# next cell calls them.
#
#   ordinal(n)                 -> 1 -> "1st", 2 -> "2nd", 10 -> "10th"
#   amount_text(amount, rng)   -> 5000 -> one of "₹5,000", "Rs 5000", "5k", ...
#   fill_values(fund, rng)     -> random amount, day and customer ID
#   make_example(kind, ...)    -> one example dictionary with its correct answer
#   make_examples(count, ...)  -> many examples in the 5 kinds, shuffled
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION: ordinal
# -----------------------------------------------------------------------------
# - 11, 12 and 13 end in "th"; otherwise the last digit decides:
#   1 -> st, 2 -> nd, 3 -> rd, anything else -> th.
def ordinal(n):
    if 11 <= n % 100 <= 13:
        return f"{n}th"
    return f"{n}" + {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")


# -----------------------------------------------------------------------------
# FUNCTION: amount_text
# -----------------------------------------------------------------------------
# - Writes the same whole-number amount in one of several styles, so the model
#   learns to turn all of them into the plain number.
# - {amount:,} adds commas: 5000 -> "5,000".
# - "5k" is only used for whole thousands (5000, not 5500).
def amount_text(amount, rng):
    styles = [f"₹{amount:,}", f"Rs {amount}", f"{amount} rupees", f"INR {amount:,}", f"{amount}"]
    if amount % 1000 == 0:
        styles.append(f"{amount // 1000}k")
    return rng.choice(styles)


# -----------------------------------------------------------------------------
# FUNCTION: fill_values
# -----------------------------------------------------------------------------
# - Picks random values for one example and returns them in a dictionary.
# - With a 20% chance, the fund name is written in lower case, the way people
#   often type. The correct answer then uses that same lower-case text,
#   because the tool description says "as written by the user".
def fill_values(fund, rng):
    if rng.random() < 0.2:
        fund = fund.lower()
    amount = rng.randrange(500, 50001, 500)   # 500, 1000, 1500, ... 50000
    day = rng.randint(1, 28)                  # 1 to 28
    return {
        "fund": fund,                             # fund name as it appears in the request
        "amount_value": amount,                   # the whole number (goes in the answer)
        "amount": amount_text(amount, rng),       # the amount as written in the request
        "amount_clean": f"₹{amount:,}",           # the amount as the assistant writes it back
        "day_value": day,                         # the whole number (goes in the answer)
        "day": ordinal(day),                      # "10th"
        "day_num": str(day),                      # "10"
        "cid": f"C{rng.randint(1001, 9999)}",     # a customer ID such as "C4821"
    }


# -----------------------------------------------------------------------------
# FUNCTION: make_example
# -----------------------------------------------------------------------------
# - kind      : one of "get_nav", "get_portfolio", "start_sip", "missing", "general"
# - patterns  : a dictionary {kind: list of patterns} (TRAIN or CHECK lists)
# - fund_list : TRAIN_FUND_NAMES or CHECK_FUND_NAMES
# - rng       : the random-number generator
def make_example(kind, patterns, fund_list, rng):

    # Random values for this example.
    v = fill_values(rng.choice(fund_list), rng)

    # .format(**v) replaces every {placeholder} in the pattern with the value
    # of the same name from v. Placeholders a pattern does not use are ignored.

    if kind == "get_nav":
        user = rng.choice(patterns[kind]).format(**v)
        return {"kind": kind, "user": user, "tool": "get_nav",
                "arguments": {"fund_name": v["fund"]}, "reply": None}

    if kind == "get_portfolio":
        user = rng.choice(patterns[kind]).format(**v)
        return {"kind": kind, "user": user, "tool": "get_portfolio",
                "arguments": {"customer_id": v["cid"]}, "reply": None}

    if kind == "start_sip":
        user = rng.choice(patterns[kind]).format(**v)
        # The answer holds whole NUMBERS, not the text written in the request.
        return {"kind": kind, "user": user, "tool": "start_sip",
                "arguments": {"fund_name": v["fund"], "amount": v["amount_value"], "day_of_month": v["day_value"]},
                "reply": None}

    if kind == "missing":
        # A pair: (request pattern, question pattern). Fill both with the same values.
        request_pattern, reply_pattern = rng.choice(patterns[kind])
        return {"kind": kind, "user": request_pattern.format(**v), "tool": None,
                "arguments": None, "reply": reply_pattern.format(**v)}

    if kind == "general":
        # A fixed (question, answer) pair; no values to fill in.
        question, answer = rng.choice(patterns[kind])
        return {"kind": kind, "user": question, "tool": None, "arguments": None, "reply": answer}

    # Any other kind name is a mistake in our own code: stop with a clear message.
    raise ValueError(f"Unknown kind: {kind}")


# -----------------------------------------------------------------------------
# THE MIX OF KINDS (shares of the total)
# -----------------------------------------------------------------------------
KIND_MIX = {"get_nav": 0.27, "start_sip": 0.27, "get_portfolio": 0.16, "missing": 0.16, "general": 0.14}


# -----------------------------------------------------------------------------
# FUNCTION: make_examples
# -----------------------------------------------------------------------------
# - Makes "count" examples split according to KIND_MIX, then shuffles them.
def make_examples(count, patterns, fund_list, rng):
    examples = []
    kinds = list(KIND_MIX)
    for i, kind in enumerate(kinds):
        # Every kind gets its share, rounded. The LAST kind gets whatever is
        # left, so the total is exactly "count".
        if i < len(kinds) - 1:
            n = round(count * KIND_MIX[kind])
        else:
            n = count - len(examples)
        examples += [make_example(kind, patterns, fund_list, rng) for _ in range(n)]
    rng.shuffle(examples)
    return examples

## 4.4 Build the training and checking sets

**Predict first:** with 1,000 training examples and the 27/27/16/16/14 mix, how many training examples need `start_sip`? (Answer: 270.)

In [17]:
# =============================================================================
# CELL: 4.4 — BUILD THE TRAINING AND CHECKING EXAMPLES
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Groups the TRAIN patterns and the CHECK patterns by kind.
# 2. Generates NUM_TRAIN_EXAMPLES training examples from TRAIN patterns and
#    TRAIN fund names.
# 3. Generates NUM_CHECK_EXAMPLES checking examples from CHECK patterns and
#    CHECK fund names. These share no wording and no fund names with training.
# 4. Prints how many of each kind, and shows one example of each kind.
# =============================================================================


# -----------------------------------------------------------------------------
# The patterns, grouped by kind
# -----------------------------------------------------------------------------
TRAIN_PATTERNS = {"get_nav": NAV_TRAIN, "get_portfolio": PORTFOLIO_TRAIN, "start_sip": SIP_TRAIN,
                  "missing": MISSING_TRAIN, "general": GENERAL_TRAIN}
CHECK_PATTERNS = {"get_nav": NAV_CHECK, "get_portfolio": PORTFOLIO_CHECK, "start_sip": SIP_CHECK,
                  "missing": MISSING_CHECK, "general": GENERAL_CHECK}


# -----------------------------------------------------------------------------
# Generate both sets
# -----------------------------------------------------------------------------
train_examples = make_examples(NUM_TRAIN_EXAMPLES, TRAIN_PATTERNS, TRAIN_FUND_NAMES, rng)
check_examples = make_examples(NUM_CHECK_EXAMPLES, CHECK_PATTERNS, CHECK_FUND_NAMES, rng)


# -----------------------------------------------------------------------------
# Count the kinds in each set
# -----------------------------------------------------------------------------
# - pd.Series([...]).value_counts() counts how often each value appears.
counts = pd.DataFrame({
    "training": pd.Series([e["kind"] for e in train_examples]).value_counts(),
    "checking": pd.Series([e["kind"] for e in check_examples]).value_counts(),
})
print(counts, "\n")


# -----------------------------------------------------------------------------
# Show one training example of each kind
# -----------------------------------------------------------------------------
for kind in KIND_MIX:
    # next(...) gives the FIRST example of this kind.
    example = next(e for e in train_examples if e["kind"] == kind)
    print(f"[{kind}]")
    print("  user     :", example["user"])
    if example["tool"]:
        # json.dumps shows the arguments exactly as JSON. Numbers have no quotes.
        print("  correct  : call", example["tool"], json.dumps(example["arguments"], ensure_ascii=False))
    else:
        print("  correct  : no tool; reply:", example["reply"])
    print()

               training  checking
get_nav             270        27
start_sip           270        27
missing             160        16
get_portfolio       160        16
general             140        14 

[get_nav]
  user     : Kotak MSCI India ETF NAV?
  correct  : call get_nav {"fund_name": "Kotak MSCI India ETF"}

[start_sip]
  user     : Kotak FMP Series 171 (1099 Days) mein Rs 9000 ki SIP shuru karo, har mahine 18 tareekh ko.
  correct  : call start_sip {"fund_name": "Kotak FMP Series 171 (1099 Days)", "amount": 9000, "day_of_month": 18}

[get_portfolio]
  user     : Show holdings for customer C1675.
  correct  : call get_portfolio {"customer_id": "C1675"}

[missing]
  user     : What funds do I currently hold?
  correct  : no tool; reply: Could you share your customer ID so I can fetch your portfolio?

[general]
  user     : What is a liquid fund?
  correct  : no tool; reply: A liquid fund invests in very short-term debt, with maturities of up to 91 days. It is often used to par

**Try it:** add a cell with `train_examples[:10]` to see ten raw examples, or change `SEED` in Part 1.3 and re-run to get different examples.

<details>
<summary><b>Check yourself:</b> why does a "missing" example have <code>tool = None</code>, even though the user wants a SIP? (click to open)</summary>

Because a required value (amount or day) is missing. Calling `start_sip` would mean the model inventing that value. The correct behaviour is to ask the user, so the correct answer is a question, not a tool call.
</details>

---
# Part 5 — Before: score the untouched model

## What we are doing in this part

Before training, we measure how well the untouched Qwen3 handles our checking requests. Four steps:

1. **5.1** Write `ask`: send one request (with the tool descriptions) to a model and get its raw text back.
2. **5.2** Write `parse_output`: read that raw text and work out whether it is a tool call (and which one, with which values) or a plain text answer. We test it on hand-written outputs first.
3. **5.3** Write the scoring: compare each output with the known correct answer, and add up the 6 scores.
4. **5.4** Run all checking requests through the untouched model and show its scores.

**What goes in:** the untouched model and the checking examples.
**What comes out:** `before_scores`, the "before" numbers.

## 5.1 Asking the model

`ask` does four things:

1. **Build** the full prompt: system instruction + tool descriptions + request, with thinking switched off.
2. **Convert** it to token IDs on the GPU.
3. **Generate:** the model writes new tokens one at a time, until it writes the stop token `<|im_end|>` or reaches 128 new tokens.
4. **Decode:** turn only the new tokens back into text.

We use `do_sample=False`: the model always picks its single most likely next token, so the same request always gives the same answer. This keeps "before" and "after" comparable.

In [18]:
# =============================================================================
# CELL: 5.1 — THE "ask" FUNCTION
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Defines two functions. Running the cell prints nothing.
#
#   generate_from_prompt(model, prompt) -> lets the model continue a prompt
#                                          and returns only the new text
#   ask(model, user_message)            -> builds the full prompt with our
#                                          system instruction and tools,
#                                          then calls generate_from_prompt
#
# ABOUT WARNINGS YOU MAY SEE WHEN THESE RUN
# -----------------------------------------
# Qwen3's own settings suggest random sampling (temperature, top_p, top_k).
# We switch sampling off, so transformers may print that those settings are
# ignored. That is expected and harmless.
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION: generate_from_prompt
# -----------------------------------------------------------------------------
def generate_from_prompt(model, prompt, max_new_tokens=128):
    """Let the model continue the prompt text, and return only the new text."""

    # Text -> token IDs, as a PyTorch tensor ("pt"), moved to the model's GPU.
    # add_special_tokens=False: the prompt already contains all special tokens.
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)

    # torch.no_grad(): we are only answering, not training, so skip the
    # records training needs. Saves memory and time.
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,                             # the prompt's token IDs and attention mask
            max_new_tokens=max_new_tokens,        # hard limit on the answer length
            do_sample=False,                      # always pick the most likely token: same answer every run
            pad_token_id=tokenizer.pad_token_id,  # which ID is padding
        )

    # The output = prompt tokens + new tokens. Keep only the new ones:
    #   inputs["input_ids"].shape[1] is the number of prompt tokens.
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]

    # Token IDs -> text. skip_special_tokens=True removes <|im_end|>.
    # (<tool_call> is NOT a special token, so it stays in the text.)
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


# -----------------------------------------------------------------------------
# FUNCTION: ask
# -----------------------------------------------------------------------------
def ask(model, user_message, max_new_tokens=128):
    """Send one user request, with our system instruction and tools, and return the raw answer text."""

    # The conversation: our standing instruction + the user's request.
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message}]

    # The full prompt, exactly as in Part 3.2.
    prompt = tokenizer.apply_chat_template(messages, tools=TOOLS, tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)

    return generate_from_prompt(model, prompt, max_new_tokens)

## 5.2 Reading the model's output

### What we are doing here

The model's answer is just text. Our code must decide which of three cases it is:

| Case | What the text looks like | `parse_output` returns |
|---|---|---|
| **Tool call** | `<tool_call>{"name": "get_nav", "arguments": {"fund_name": "X"}}</tool_call>` | `{"type": "tool", "name": "get_nav", "arguments": {"fund_name": "X"}}` |
| **Plain answer** | `An SIP is ...` (no `<tool_call>` at all) | `{"type": "text", "text": "An SIP is ..."}` |
| **Broken call** | `<tool_call>{"name": "get_nav", "arguments": {"fund_` (cut off or not valid JSON) | `{"type": "broken", "text": ...}` |

We test it on four hand-written outputs before using it on the model. Checking our own scoring code first matters: if it is wrong, every score after it is wrong too.

In [19]:
# =============================================================================
# CELL: 5.2 — READ THE MODEL'S OUTPUT (AND TEST THE READER)
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Defines parse_output(text), which turns the model's raw text into one of:
#        {"type": "tool", "name": ..., "arguments": {...}}
#        {"type": "text", "text": ...}
#        {"type": "broken", "text": ...}
# 2. Tests it on 4 hand-written outputs, so we can see it works before
#    trusting it with the model.
# =============================================================================


def parse_output(text):
    """Turn the model's raw answer into a tool call, a plain text answer, or 'broken'."""

    # Remove any <think> ... </think> block, in case the model writes one.
    #   re.sub(pattern, "", text) deletes every part that matches the pattern.
    #   ".*?" means "any characters, as few as possible".
    #   flags=re.DOTALL lets "." also match line breaks.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # CASE 2: no "<tool_call>" anywhere -> a plain text answer.
    if "<tool_call>" not in text:
        return {"type": "text", "text": text}

    # Find the text between <tool_call> and </tool_call>.
    #   (.*?) "captures" that middle part so we can read it with .group(1).
    #   \s* allows any spaces or line breaks around it.
    match = re.search(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.DOTALL)

    # The opening tag is there but no closing tag: the call was cut off.
    if match is None:
        return {"type": "broken", "text": text}

    # Try to read the middle part as JSON.
    try:
        call = json.loads(match.group(1))
    # json.JSONDecodeError = the text is not valid JSON.
    except json.JSONDecodeError:
        return {"type": "broken", "text": text}

    # Valid JSON, but not in the form {"name": ..., "arguments": {...}}.
    if not isinstance(call, dict) or "name" not in call:
        return {"type": "broken", "text": text}

    # CASE 1: a proper tool call. "arguments" defaults to {} if missing.
    return {"type": "tool", "name": call["name"], "arguments": call.get("arguments", {})}


# -----------------------------------------------------------------------------
# TEST THE READER ON 4 HAND-WRITTEN OUTPUTS
# -----------------------------------------------------------------------------
test_outputs = [
    '<tool_call>\n{"name": "get_nav", "arguments": {"fund_name": "SBI Credit Risk Fund"}}\n</tool_call>',  # a good tool call
    "An SIP is a way to invest a fixed amount every month.",                                             # a plain answer
    '<tool_call>\n{"name": "start_sip", "arguments": {"fund_name": "X", "amount": 50',                    # cut off
    '<tool_call>\n{name: get_nav}\n</tool_call>',                                                          # not valid JSON
]
for output in test_outputs:
    print(parse_output(output))

{'type': 'tool', 'name': 'get_nav', 'arguments': {'fund_name': 'SBI Credit Risk Fund'}}
{'type': 'text', 'text': 'An SIP is a way to invest a fixed amount every month.'}
{'type': 'broken', 'text': '<tool_call>\n{"name": "start_sip", "arguments": {"fund_name": "X", "amount": 50'}
{'type': 'broken', 'text': '<tool_call>\n{name: get_nav}\n</tool_call>'}


## 5.3 Scoring

### What we are doing here

For each checking example we know the correct answer, so we can judge the model's output:

| Kind of example | Counted as correct when... |
|---|---|
| `get_nav`, `get_portfolio`, `start_sip` | the output is a tool call, with the **right tool name** and **exactly the right values**: same value names, numbers as whole numbers, text the same apart from capital letters and spaces at the ends |
| `general` | the output is a non-empty plain-text answer, with **no tool call** |
| `missing` | the output is plain text, with **no tool call**, and it **asks a question** (contains "?") |

For `general` and `missing` we do **not** score the wording of the answer, only the behaviour. Scoring free text fairly is a much harder problem, and the behaviour is what our code depends on.

Then we add up the 6 scores from the introduction.

In [20]:
# =============================================================================
# CELL: 5.3 — JUDGE ONE OUTPUT, AND SCORE A MODEL ON ALL CHECKING EXAMPLES
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Defines four functions. Running the cell prints nothing.
#
#   arguments_match(got, expected) -> are the model's values exactly right?
#   judge(example, parsed)         -> marks for one example
#   score_model(model, examples)   -> asks the model every example, judges
#                                     each answer, returns all results
#   summarise(results)             -> turns the results into the 6 scores (%)
# =============================================================================


# Kinds whose correct answer is a tool call.
TOOL_KINDS = {"get_nav", "get_portfolio", "start_sip"}


# -----------------------------------------------------------------------------
# FUNCTION: arguments_match
# -----------------------------------------------------------------------------
def arguments_match(got, expected):
    """True only if 'got' has exactly the expected value names and every value is right."""

    # Must be a dictionary with exactly the same value names. No extras, none missing.
    if not isinstance(got, dict) or set(got) != set(expected):
        return False

    for name, correct_value in expected.items():
        value = got[name]

        # Whole-number values (amount, day_of_month) must be real whole
        # numbers. "5000" (text) or 5000.0 (decimal) count as wrong, because
        # the tool description asks for an integer.
        # (bool is excluded because Python treats True as the number 1.)
        if isinstance(correct_value, int):
            if isinstance(value, bool) or not isinstance(value, int) or value != correct_value:
                return False

        # Text values (fund_name, customer_id): same text, ignoring capital
        # letters and spaces at the ends.
        else:
            if not isinstance(value, str) or value.strip().casefold() != correct_value.strip().casefold():
                return False

    return True


# -----------------------------------------------------------------------------
# FUNCTION: judge
# -----------------------------------------------------------------------------
def judge(example, parsed):
    """Return the marks for one checking example."""

    # For tool kinds:
    if example["kind"] in TOOL_KINDS:
        readable = parsed["type"] == "tool"                                   # our code could read a tool call
        right_tool = readable and parsed["name"] == example["tool"]            # ...and it is the right tool
        right_args = right_tool and arguments_match(parsed["arguments"], example["arguments"])  # ...with the right values
        return {"readable": readable, "right_tool": right_tool, "correct": right_args}

    # For general questions: correct if there is no tool call AND some answer
    # text (an empty reply does not count).
    if example["kind"] == "general":
        return {"correct": parsed["type"] == "text" and parsed["text"] != ""}

    # For missing details: no tool call AND a question is asked.
    return {"correct": parsed["type"] == "text" and "?" in parsed["text"]}


# -----------------------------------------------------------------------------
# FUNCTION: score_model
# -----------------------------------------------------------------------------
def score_model(model, examples):
    """Ask the model every example, judge each answer, and return the list of results."""

    results = []
    start = time.time()  # remember the start time

    for i, example in enumerate(examples, start=1):
        raw = ask(model, example["user"])   # the model's raw text
        parsed = parse_output(raw)          # tool call / text / broken
        marks = judge(example, parsed)      # the marks for this example

        # Keep everything together, to look at mistakes later.
        # {**a, **b} merges two dictionaries into one.
        results.append({**example, "raw": raw, "parsed": parsed, **marks})

        # A progress line every 10 examples.
        if i % 10 == 0:
            print(f"  {i}/{len(examples)} done ({time.time() - start:.0f} seconds so far)")

    return results


# -----------------------------------------------------------------------------
# FUNCTION: summarise
# -----------------------------------------------------------------------------
def summarise(results):
    """Turn the list of results into the 6 scores, as percentages."""

    # A helper: the share of True values in a list, as a percentage.
    # Returns 0 if the list is empty, to avoid dividing by zero.
    def pct(values):
        return round(100 * sum(values) / len(values), 1) if values else 0.0

    tool_results = [r for r in results if r["kind"] in TOOL_KINDS]
    return {
        "Readable tool call (%)":           pct([r["readable"] for r in tool_results]),
        "Correct tool (%)":                 pct([r["right_tool"] for r in tool_results]),
        "Correct tool and arguments (%)":   pct([r["correct"] for r in tool_results]),
        "No tool when none needed (%)":     pct([r["correct"] for r in results if r["kind"] == "general"]),
        "Asked for missing details (%)":    pct([r["correct"] for r in results if r["kind"] == "missing"]),
        "Overall correct (%)":              pct([r["correct"] for r in results]),
    }

## 5.4 Score the untouched model

### What we are doing here

We run every checking request through the untouched Qwen3 and score the answers. With 100 checking examples, my estimate is about 5 minutes on a T4. If you need it faster for a live session, lower `NUM_CHECK_EXAMPLES` to 50 in Part 1.3.

**Predict first:** Qwen3 already knows tool calling. Which score do you expect to be **lowest** before training: correct tool, correct arguments, no tool when none needed, or asking for missing details?

In [21]:
# =============================================================================
# CELL: 5.4 — SCORE THE UNTOUCHED MODEL ("BEFORE")
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Asks the untouched model every checking request and scores the answers.
# 2. Shows the 6 scores.
# 3. Shows up to 4 examples it got wrong, with the correct answer and the
#    model's raw output, so we can see WHAT kind of mistakes it makes.
# =============================================================================


# -----------------------------------------------------------------------------
# Score
# -----------------------------------------------------------------------------
print(f"Scoring the untouched model on {len(check_examples)} checking examples...")
before_results = score_model(model, check_examples)
before_scores = summarise(before_results)


# -----------------------------------------------------------------------------
# Show the scores as a small table
# -----------------------------------------------------------------------------
# - pd.Series(dictionary, name=...) shows a dictionary as a one-column table.
print()
print(pd.Series(before_scores, name="BEFORE"))


# -----------------------------------------------------------------------------
# Show up to 4 mistakes
# -----------------------------------------------------------------------------
# - [r for r in before_results if not r["correct"]] -> the wrong ones.
# - [:4] -> at most 4 of them.
print("\nSome mistakes from the untouched model:\n")
for r in [r for r in before_results if not r["correct"]][:4]:
    print(f"[{r['kind']}]  {r['user']}")
    if r["tool"]:
        print("  correct   : call", r["tool"], json.dumps(r["arguments"], ensure_ascii=False))
    else:
        print("  correct   : no tool;", r["reply"])
    # [:300] -> show at most 300 characters of the raw output.
    print("  model said:", r["raw"][:300])
    print()

Scoring the untouched model on 100 checking examples...
  10/100 done (69 seconds so far)
  20/100 done (95 seconds so far)
  30/100 done (123 seconds so far)
  40/100 done (160 seconds so far)
  50/100 done (221 seconds so far)
  60/100 done (260 seconds so far)
  70/100 done (291 seconds so far)
  80/100 done (322 seconds so far)
  90/100 done (360 seconds so far)
  100/100 done (398 seconds so far)

Readable tool call (%)             48.6
Correct tool (%)                   48.6
Correct tool and arguments (%)     47.1
No tool when none needed (%)      100.0
Asked for missing details (%)      18.8
Overall correct (%)                50.0
Name: BEFORE, dtype: float64

Some mistakes from the untouched model:

[start_sip]  41500 rupees har mahine quant dynamic asset allocation fund mein daalna hai, 15th ko.
  correct   : call start_sip {"fund_name": "quant dynamic asset allocation fund", "amount": 41500, "day_of_month": 15}
  model said: I need to start a monthly SIP of ₹41500 in the "Qua

Read the mistakes above. Common patterns to look for:

- **Numbers as text:** amounts written as text (`"5000"`) or left as `"5k"`.
- **Fund names changed:** the name rewritten instead of copied.
- **Calls when it should ask:** `start_sip` called with an invented amount or day, instead of asking.
- **Calls with no need:** a tool called for a general question.

Training targets exactly these.

---
# Part 6 — Prepare the examples for training

## What we are doing in this part

Each example becomes one complete conversation in Qwen3's layout, and then token IDs, with a mark on every token saying **"learn this"** or **"ignore this"**.

1. **6.1** Turn each example into a full conversation: system instruction + tools + request + the **correct answer**.
2. **6.2** Split it into **Part A** (everything up to the answer) and **Part B** (the answer). Part A tokens get the label `-100` ("ignore"); Part B tokens are learned.
3. **6.3** Put everything into `Dataset` tables for the trainer, and look at one example's two parts.

### The labels, with a small example

| Token | `…user\nNAV of X?` | `<\|im_end\|>` | `<\|im_start\|>assistant` | `<tool_call>` | `{"name": "get_nav"…` | `</tool_call>` | `<\|im_end\|>` |
|---|---|---|---|---|---|---|---|
| Part | A | A | A | **B** | **B** | **B** | **B** |
| Label | −100 | −100 | −100 | learned | learned | learned | learned |

The final `<|im_end|>` (the stop token) is in Part B, so the model learns to **stop** right after its answer.

In [22]:
# =============================================================================
# CELL: 6.1 + 6.2 — BUILD EACH TRAINING CONVERSATION AND ITS "LEARN / IGNORE" LABELS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Defines two functions:
#
#   full_conversation(example) -> the conversation as a list of messages,
#                                 ending with the CORRECT assistant answer
#   to_features(example)       -> input_ids, attention_mask and labels:
#                                   input_ids      = Part A tokens + Part B tokens
#                                   labels         = -100 for every Part A token,
#                                                    the token ID for every Part B token
#                                   attention_mask = 1 for every token
#
# HOW PART A AND PART B ARE FOUND
# -------------------------------
#   prompt_text = the prompt exactly as at question time (Part 3.2)
#   full_text   = the whole conversation including the correct answer
# full_text always starts with prompt_text, so:
#   answer_text = full_text with prompt_text cut off the front
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION: full_conversation
# -----------------------------------------------------------------------------
def full_conversation(example):
    """The conversation for one example, ending with the correct assistant answer."""

    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": example["user"]}]

    if example["tool"]:
        # The correct answer is a tool call. In the standard message format, a
        # tool call goes in "tool_calls", with empty "content". The chat
        # template turns this into <tool_call>{...}</tool_call>.
        messages.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function",
                                         "function": {"name": example["tool"], "arguments": example["arguments"]}}]})
    else:
        # The correct answer is plain text (a general answer or a question).
        messages.append({"role": "assistant", "content": example["reply"]})

    return messages


# -----------------------------------------------------------------------------
# FUNCTION: to_features
# -----------------------------------------------------------------------------
def to_features(example):
    """Token IDs plus 'learn / ignore' labels for one example."""

    messages = full_conversation(example)

    # Part A as text: everything except the answer, ending exactly like the
    # prompt the model sees at question time (with the empty <think> block).
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tools=TOOLS, tokenize=False,
                                                add_generation_prompt=True, enable_thinking=False)

    # The whole conversation as text, including the answer.
    full_text = tokenizer.apply_chat_template(messages, tools=TOOLS, tokenize=False)

    # Safety check: the whole text must start with Part A. If not, the
    # split below would be wrong, so stop with a clear message.
    assert full_text.startswith(prompt_text), "The prompt is not a prefix of the full conversation."

    # Part B as text: cut Part A off the front.
    # .rstrip("\n") removes the final line break after <|im_end|>, so the
    # stop token is the last thing the model learns.
    answer_text = full_text[len(prompt_text):].rstrip("\n")

    # Both parts -> token IDs. add_special_tokens=False: add nothing extra.
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]

    return {
        "input_ids": prompt_ids + answer_ids,                     # the model reads all of it
        "attention_mask": [1] * (len(prompt_ids) + len(answer_ids)),  # 1 = a real token (not padding)
        "labels": [-100] * len(prompt_ids) + answer_ids,          # learn only Part B
    }

In [23]:
# =============================================================================
# CELL: 6.3 — BUILD THE TRAINING AND CHECKING TABLES, AND LOOK AT ONE EXAMPLE
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Runs to_features on every training and checking example.
# 2. Drops any example longer than MAX_TOKENS (in practice none are).
# 3. Puts them into Dataset tables, which the trainer needs.
# 4. Shows one training example split into its two parts, so you can see
#    exactly which text is learned.
# =============================================================================


# -----------------------------------------------------------------------------
# Convert every example, keeping only those within MAX_TOKENS
# -----------------------------------------------------------------------------
train_features = [f for f in (to_features(e) for e in train_examples) if len(f["input_ids"]) <= MAX_TOKENS]
check_features = [f for f in (to_features(e) for e in check_examples) if len(f["input_ids"]) <= MAX_TOKENS]

# -----------------------------------------------------------------------------
# Put them into Dataset tables
# -----------------------------------------------------------------------------
# - Dataset.from_list(list of dictionaries) makes a table with one row per
#   dictionary and one column per key (input_ids, attention_mask, labels).
train_dataset = Dataset.from_list(train_features)
check_dataset = Dataset.from_list(check_features)

print(f"Training rows: {len(train_dataset)} of {len(train_examples)}")
print(f"Checking rows: {len(check_dataset)} of {len(check_examples)}")

# The average length, to see how much of each example is Part A.
lengths = [len(f["input_ids"]) for f in train_features]
learned = [sum(1 for label in f["labels"] if label != -100) for f in train_features]
print(f"Average tokens per example: {sum(lengths) / len(lengths):.0f}, of which learned (Part B): {sum(learned) / len(learned):.0f}")


# -----------------------------------------------------------------------------
# Look at one example: which part is ignored, which part is learned
# -----------------------------------------------------------------------------
sample = train_features[0]

# Split the token IDs by their label.
ignored_ids = [t for t, label in zip(sample["input_ids"], sample["labels"]) if label == -100]
learned_ids = [t for t, label in zip(sample["input_ids"], sample["labels"]) if label != -100]

# [-150:] -> show only the LAST 150 characters of Part A (the tool
# descriptions before that are long).
print("\n----- PART A (ignored), last 150 characters -----")
print(tokenizer.decode(ignored_ids)[-150:])
print("\n----- PART B (learned) -----")
print(tokenizer.decode(learned_ids))

Training rows: 1000 of 1000
Checking rows: 100 of 100
Average tokens per example: 530, of which learned (Part B): 33

----- PART A (ignored), last 150 characters -----
"arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Kotak MSCI India ETF NAV?<|im_end|>
<|im_start|>assistant
<think>

</think>



----- PART B (learned) -----
<tool_call>
{"name": "get_nav", "arguments": {"fund_name": "Kotak MSCI India ETF"}}
</tool_call><|im_end|>


Check the output:

- Part A ends with `<|im_start|>assistant` and the empty `<think>` block.
- Part B is exactly the correct answer, ending with `<|im_end|>`.

Look at the average: most of each example is Part A. If Part A counted towards the loss, it would dominate it, because it is almost the same text every time. With labels, the loss only measures the answers.

---
# Part 7 — Attach LoRA

## What we are doing in this part

Exactly the same method as the TinyLlama notebook:

1. **Freeze** all 1.72 billion original numbers.
2. **Attach** a small trainable pair of tables (A and B) next to 7 tables in each of Qwen3's **28 layers** (196 pairs).
3. **Print** how many numbers can now be trained.

### Where the trainable count comes from (rank 16)

Each LoRA pair adds `16 × (rows + columns)` numbers. Qwen3-1.7B's table sizes:

| Table | Size | LoRA numbers |
|---|---|---|
| `q_proj` | 2,048 × 2,048 | 16 × 4,096 = 65,536 |
| `k_proj` | 2,048 × 1,024 | 16 × 3,072 = 49,152 |
| `v_proj` | 2,048 × 1,024 | 16 × 3,072 = 49,152 |
| `o_proj` | 2,048 × 2,048 | 65,536 |
| `gate_proj` | 2,048 × 6,144 | 16 × 8,192 = 131,072 |
| `up_proj` | 2,048 × 6,144 | 131,072 |
| `down_proj` | 6,144 × 2,048 | 131,072 |
| **One layer** | | **622,592** |
| **28 layers** | | **17,432,576** |

That is about **1%** of the model. Expect the next cell to print roughly that number.

At the start, every B table is all zeros. So the model still behaves exactly like the untouched model, until training changes A and B.

In [24]:
# =============================================================================
# CELL: FREEZE THE MODEL AND ATTACH LoRA
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Switches off the generation cache (only useful when generating).
# 2. Freezes all original numbers and prepares the 4-bit model for training.
# 3. Writes down the LoRA settings.
# 4. Attaches an A-and-B pair next to each of the 7 table types in all 28
#    layers.
# 5. Prints the number of trainable numbers.
#
# WHAT YOU SHOULD SEE (by my calculation)
# ---------------------------------------
#     trainable params: 17,432,576 || all params: about 1,738,000,000 || trainable%: about 1.0
# =============================================================================


# -----------------------------------------------------------------------------
# LINE 1: switch off the generation cache
# -----------------------------------------------------------------------------
# - The cache speeds up GENERATING (it remembers earlier tokens' results).
#   Training does not use it, so we switch it off now and back on in Part 9.
model.config.use_cache = False

# -----------------------------------------------------------------------------
# LINE 2: freeze and prepare the 4-bit model
# -----------------------------------------------------------------------------
# - Freezes every original number, moves a few small sensitive parts to
#   32-bit for stable training, and lets the LoRA tables be trained.
# - use_gradient_checkpointing=False: that option saves memory but slows
#   training. The T4 has enough memory without it.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

# -----------------------------------------------------------------------------
# LINE 3: the LoRA settings
# -----------------------------------------------------------------------------
lora_config = LoraConfig(
    r=LORA_R,                 # rank: 16 patterns per change table
    lora_alpha=LORA_R * 2,    # the correction is multiplied by alpha / r = 2
    lora_dropout=0.05,        # while training, randomly switch off 5% of LoRA inputs (helps against memorising)
    # The 7 table types in each layer:
    #   q_proj, k_proj, v_proj, o_proj -> the attention part (which earlier tokens matter)
    #   gate_proj, up_proj, down_proj  -> the feed-forward part (processes each token further)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",              # do not train the small "bias" lists
    task_type="CAUSAL_LM",    # this model predicts the next token
)

# -----------------------------------------------------------------------------
# LINE 4: attach LoRA
# -----------------------------------------------------------------------------
# - Finds every table named above (7 x 28 = 196) and attaches an A-and-B pair.
model = get_peft_model(model, lora_config)

# -----------------------------------------------------------------------------
# LINE 5: show how many numbers can be trained
# -----------------------------------------------------------------------------
model.print_trainable_parameters()

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


---
# Part 8 — Train

## What we are doing in this part

1. **8.1** Write down the training settings, the batch builder, and the training loop.
2. **8.2** Measure the starting loss on the checking examples.
3. **8.3** Train.
4. **8.4** Chart the loss.

### What happens in one training step

1. Take a **batch** of examples.
2. For every **Part B** token, the model predicts it from all the tokens before it.
3. The **loss** measures how wrong those predictions were (lower is better). Part A tokens are ignored.
4. Work out which way to nudge each LoRA number to lower the loss.
5. Nudge them by a small amount (the **learning rate**).

### The training settings

| Setting | Value | Meaning |
|---|---|---|
| `per_device_train_batch_size` | 4 | 4 examples on the GPU at a time |
| `gradient_accumulation_steps` | 4 | collect 4 batches before one nudge: 16 examples per nudge |
| `num_train_epochs` | 1 | go through the training examples once |
| `learning_rate` | 0.0002 | the size of each nudge |
| `lr_scheduler_type` | cosine | nudges get gradually smaller towards the end |
| `warmup_steps` | 5 | the learning rate grows from 0 over the first 5 nudges |
| `logging_steps` / `eval_steps` | 5 / 20 | show the training loss every 5 nudges; measure the checking loss every 20 |
| `fp16` | True | 16-bit maths, faster on a T4 |

With 1,000 examples and 16 per nudge, one epoch is about **63 nudges** (steps).

### The batch builder

We use `DataCollatorForSeq2Seq`. For each batch it:

- pads the shorter examples with the padding token, so all are the same length;
- pads their **labels** with `-100`, so the padding is also ignored.

The TinyLlama notebook used a different batch builder, because it created the labels itself. Here we made our own labels in Part 6, so we need one that keeps them.

In [25]:
# =============================================================================
# CELL: 8.1 — TRAINING SETTINGS, BATCH BUILDER AND TRAINING LOOP
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Creates three things. Nothing trains yet, and nothing is printed.
#   training_args -> every training setting
#   data_collator -> builds each batch: pads tokens and labels
#   trainer       -> the ready-made training loop
# =============================================================================


training_args = TrainingArguments(
    output_dir="training-output",       # a folder the trainer needs for its own files
    num_train_epochs=NUM_EPOCHS,        # passes over the training examples (1)
    per_device_train_batch_size=4,      # examples on the GPU at one time
    gradient_accumulation_steps=4,      # 4 batches per nudge -> 16 examples per nudge
    learning_rate=LEARNING_RATE,        # size of each nudge (0.0002)
    lr_scheduler_type="cosine",         # nudges shrink gradually towards the end
    warmup_steps=5,                     # grow the learning rate from 0 over the first 5 nudges
    logging_steps=5,                    # show the training loss every 5 nudges
    eval_strategy="steps",              # measure the checking loss at fixed step intervals...
    eval_steps=20,                      # ...every 20 nudges
    per_device_eval_batch_size=4,       # checking batch size
    save_strategy="no",                 # no copies during training; we save in Part 11
    fp16=True,                          # 16-bit maths for speed on the T4
    report_to="none",                   # keep all numbers inside this notebook
    seed=SEED,                          # same random choices on every run
)

# -----------------------------------------------------------------------------
# The batch builder
# -----------------------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,       # knows the padding token
    padding=True,              # pad every example to the longest one in its batch
    label_pad_token_id=-100,   # pad the labels with -100 ("ignore")
)

# -----------------------------------------------------------------------------
# The training loop
# -----------------------------------------------------------------------------
trainer = Trainer(
    model=model,                  # Qwen3 with LoRA attached
    args=training_args,           # the settings above
    train_dataset=train_dataset,  # the examples to learn from
    eval_dataset=check_dataset,   # the examples to measure the checking loss on
    data_collator=data_collator,  # how to build batches
)

## 8.2 The starting loss

The loss on the checking examples **before** any training. Because the B tables start at zero, this is the untouched model's loss, measured **only on the answer tokens**.

In [26]:
# =============================================================================
# CELL: 8.2 — LOSS ON THE CHECKING EXAMPLES BEFORE TRAINING
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# trainer.evaluate() runs the model over the checking examples without
# changing anything and returns the average loss of the Part B tokens.
# =============================================================================


loss_before = trainer.evaluate()["eval_loss"]  # the average loss, before training
print(f"Checking loss BEFORE training: {loss_before:.3f}")

Training Loss,Validation Loss,Step
No log,1.034512,0


Checking loss BEFORE training: 1.035


## 8.3 Train

Run the next cell and watch the table that appears:

- **Training Loss:** the loss on the batches being learned from.
- **Validation Loss:** the loss on the checking examples. These have new wording and new fund names, so this is the number to trust.

My estimate is roughly 15–20 minutes on a T4. **This is not yet measured.** Keep the browser tab open while it runs.

In [27]:
# =============================================================================
# CELL: 8.3 — TRAIN
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Runs the training loop: for every step, 16 examples -> predictions on the
# Part B tokens -> loss -> nudge every LoRA number. Then prints how long it
# took.
# =============================================================================


train_result = trainer.train()  # the actual training; a live progress table appears

# train_runtime is in seconds; / 60 -> minutes.
print(f"Training finished in {train_result.metrics['train_runtime'] / 60:.1f} minutes.")

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.28 GiB. GPU 0 has a total capacity of 14.56 GiB of which 569.81 MiB is free. Including non-PyTorch memory, this process has 14.00 GiB memory in use. Of the allocated memory 13.51 GiB is allocated by PyTorch, and 369.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# =============================================================================
# CELL: 8.4 — CHART THE LOSS, AND THE CHECKING LOSS AFTER TRAINING
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Reads the trainer's log of every loss it reported.
# 2. Draws the training loss and the checking loss as two lines.
# 3. Measures the checking loss once more and prints it next to the
#    "before" number.
# =============================================================================


# The trainer's log: a list of dictionaries such as {"loss": 0.4, "step": 5}
# or {"eval_loss": 0.3, "step": 20}.
history = trainer.state.log_history

# Pull out the steps and values for each line.
train_steps = [h["step"] for h in history if "loss" in h]
train_losses = [h["loss"] for h in history if "loss" in h]
check_steps = [h["step"] for h in history if "eval_loss" in h]
check_losses = [h["eval_loss"] for h in history if "eval_loss" in h]

# Draw the chart.
plt.figure(figsize=(8, 4))                                                     # 8 wide, 4 high
plt.plot(train_steps, train_losses, label="Training loss")                     # line 1
plt.plot(check_steps, check_losses, marker="o", label="Checking loss")         # line 2, with dots
plt.xlabel("Training step")
plt.ylabel("Loss (lower is better)")
plt.title("Loss during fine-tuning (answer tokens only)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# The checking loss after training.
loss_after = trainer.evaluate()["eval_loss"]
print(f"Checking loss BEFORE training: {loss_before:.3f}")
print(f"Checking loss AFTER training : {loss_after:.3f}")

A falling checking loss means the model now gives higher probability to the **exact** correct answers, on requests with wording and fund names it never trained on. The real test, though, is the scores in Part 9: whether those answers are actually right.

---
# Part 9 — After: score the trained model

## What we are doing in this part

We run the **same checking examples** through the trained model, with the same `ask`, `parse_output` and scoring, then put "before" and "after" side by side. Then we look at specific examples that changed.

**Predict first:** which score will improve the most?

In [ ]:
# =============================================================================
# CELL: SCORE THE TRAINED MODEL AND COMPARE WITH "BEFORE"
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Switches the model to answering mode.
# 2. Scores it on the same checking examples as Part 5.4.
# 3. Shows a before / after / change table.
# =============================================================================


# -----------------------------------------------------------------------------
# Answering mode
# -----------------------------------------------------------------------------
model.eval()                    # switch off training-only behaviour (LoRA dropout)
model.config.use_cache = True   # switch the generation cache back on

# -----------------------------------------------------------------------------
# Score
# -----------------------------------------------------------------------------
print(f"Scoring the trained model on {len(check_examples)} checking examples...")
after_results = score_model(model, check_examples)
after_scores = summarise(after_results)

# -----------------------------------------------------------------------------
# Before / after table
# -----------------------------------------------------------------------------
# - pd.DataFrame({column name: values}) builds a table.
comparison = pd.DataFrame({"Before": before_scores, "After": after_scores})
comparison["Change"] = comparison["After"] - comparison["Before"]   # positive = better
comparison

In [ ]:
# =============================================================================
# CELL: LOOK AT EXAMPLES THAT CHANGED
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Shows up to 4 examples that were WRONG before and are RIGHT after.
# 2. Shows up to 4 examples that are STILL wrong after training, so we stay
#    honest about what the model cannot do yet.
# =============================================================================


# -----------------------------------------------------------------------------
# FUNCTION: show_pair
# -----------------------------------------------------------------------------
# - Prints one example with the correct answer and both raw outputs.
def show_pair(before, after):
    print(f"[{after['kind']}]  {after['user']}")
    if after["tool"]:
        print("  correct:", after["tool"], json.dumps(after["arguments"], ensure_ascii=False))
    else:
        print("  correct: no tool;", after["reply"])
    print("  BEFORE :", before["raw"][:250])
    print("  AFTER  :", after["raw"][:250])
    print()


# zip(...) walks through the before and after results together, in the
# same order (the same checking examples).
pairs = list(zip(before_results, after_results))

print("=" * 30, "FIXED BY TRAINING", "=" * 30, "\n")
for before, after in [p for p in pairs if not p[0]["correct"] and p[1]["correct"]][:4]:
    show_pair(before, after)

print("=" * 30, "STILL WRONG", "=" * 30, "\n")
for before, after in [p for p in pairs if not p[1]["correct"]][:4]:
    show_pair(before, after)

**How to talk about the result:** the trained model now follows **our** conventions for **our** tools: whole-number amounts, fund names copied exactly, no tool for general questions, and asking when a detail is missing.

What it still gets wrong is worth showing too. A real system would add checks in code, such as the input checks inside `start_sip`, and a confirmation step before any real money moves.

---
# Part 10 — The live assistant

## What we are doing in this part

We connect everything into one function, `run_assistant`, and use it like a real assistant:

```
YOUR REQUEST
   │
   ▼
1. MODEL DECIDES      ask(model, request)            → <tool_call>{...}</tool_call>  or plain text
   │
   ▼
2. CODE READS         parse_output(...)              → tool name + values
   │
   ▼
3. CODE RUNS TOOL     run_tool(name, values)         → real NAV from AMFI / demo portfolio / demo SIP
   │
   ▼
4. MODEL REPLIES      the tool result is given back  → a short, friendly sentence for the user
                      to the model as a "tool" message
```

If the model answers in plain text (a general question, or asking for a missing detail), steps 2 to 4 are skipped and that text is the reply.

For step 4 the model gets the conversation **without** the tool descriptions, so it writes a reply instead of calling another tool. We did not train step 4: Qwen3 already writes good replies from a tool result.

In [ ]:
# =============================================================================
# CELL: THE run_assistant FUNCTION
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Defines run_assistant(request, model), which runs the 4 steps above and
# prints each one, so the audience can follow the decision, the tool call,
# the tool's result, and the final reply.
# =============================================================================


def run_assistant(request, model=model):
    """Model decides -> our code runs the tool -> model replies. Prints every step."""

    print("YOU      :", request)

    # STEP 1: the model decides.
    raw = ask(model, request)

    # STEP 2: our code reads the decision.
    parsed = parse_output(raw)

    # No tool call: the plain text IS the reply.
    if parsed["type"] == "text":
        print("DECISION : no tool needed")
        print("REPLY    :", parsed["text"])
        return

    # A broken tool call: show it, and stop.
    if parsed["type"] == "broken":
        print("DECISION : the model wrote a tool call our code could not read:")
        print(raw)
        return

    # A proper tool call.
    print(f"DECISION : call {parsed['name']} with {json.dumps(parsed['arguments'], ensure_ascii=False)}")

    # STEP 3: our code runs the tool.
    result = run_tool(parsed["name"], parsed["arguments"])
    # Show the result as JSON. [:800] keeps long results short on screen.
    print("TOOL     :", json.dumps(result, ensure_ascii=False)[:800])

    # STEP 4: give the result back to the model, and let it write the reply.
    # The conversation now has 4 messages:
    #   system instruction, the request, the model's tool call, the tool's result.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": request},
        {"role": "assistant", "content": "",
         "tool_calls": [{"type": "function", "function": {"name": parsed["name"], "arguments": parsed["arguments"]}}]},
        {"role": "tool", "content": json.dumps(result, ensure_ascii=False)},
    ]
    # No tools=... here, so the model writes a reply instead of another tool call.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    print("REPLY    :", generate_from_prompt(model, prompt, max_new_tokens=150))

In [ ]:
# =============================================================================
# CELL: TRY THE ASSISTANT ON 5 REQUESTS
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Runs run_assistant on 5 requests that cover every kind of behaviour:
#   a real NAV lookup, a demo portfolio, a demo SIP, a general question,
#   and a request with a missing detail.
# =============================================================================


demo_requests = [
    "What is the NAV of Axis Children's Fund?",                          # get_nav, real AMFI data
    "Show my portfolio, customer ID C1003",                               # get_portfolio, demo data
    "Start a SIP of 5k in SBI Credit Risk Fund on the 7th of every month",  # start_sip: "5k" must become 5000
    "What is the difference between Growth and IDCW?",                    # general question: no tool
    "Start a SIP in HDFC Flexi Cap Fund",                                 # missing amount and day: should ask
]

for request in demo_requests:
    run_assistant(request)
    print("=" * 80)

## Your turn

Type any request into the box on the right of the next cell and run it. Ideas:

- *"parag parikh flexi cap ka NAV kya hai?"*
- *"Put ₹2,500 every month into SBI Small Cap Fund on the 3rd"*
- *"Customer C1001, what do I own?"*
- *"Should I invest in mutual funds?"* (general: no tool)
- *"Start a SIP of ₹1,000 on the 15th"* (fund missing: should ask)

In [ ]:
# =============================================================================
# CELL: YOUR OWN REQUEST
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Sends one request of your choice through the full assistant.
# "#@param {type:"string"}" makes Colab show a text box on the right.
# =============================================================================


my_request = "parag parikh flexi cap ka NAV kya hai?"  #@param {type:"string"}

run_assistant(my_request)

---
# Part 11 — Save the adapter

## What we are doing in this part

1. **11.1** Save the trained LoRA tables (the **adapter**) and the tokenizer to a folder, and list the files.
2. **11.2** Load a fresh, untouched Qwen3, attach the saved adapter, and run one request, to prove the save worked.
3. **11.3** Optionally download the folder, or upload it to Hugging Face. Colab deletes all files when the session ends.

**Expected size:** the adapter holds about 17.4 million numbers in 32-bit, which is about 67 MB. Qwen3's tokenizer files add roughly 15 MB. Compare that with about 3.4 GB for the full model.

In [ ]:
# =============================================================================
# CELL: 11.1 — SAVE THE ADAPTER AND LIST THE FILES
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Saves only the LoRA tables and their settings into ADAPTER_DIR.
# 2. Saves the tokenizer next to them.
# 3. Lists every file with its size, and the folder total.
#
# Notice: the tool code and descriptions are NOT saved here. Anyone using
# this adapter needs the same TOOLS descriptions and the same SYSTEM_PROMPT
# that it was trained with.
# =============================================================================


model.save_pretrained(ADAPTER_DIR)      # the A and B tables + adapter_config.json
tokenizer.save_pretrained(ADAPTER_DIR)  # the tokenizer files

total_bytes = 0
for file_name in sorted(os.listdir(ADAPTER_DIR)):                         # every file, alphabetical
    file_bytes = os.path.getsize(os.path.join(ADAPTER_DIR, file_name))    # its size in bytes
    total_bytes += file_bytes                                             # add to the total
    print(f"{file_name:35s} {file_bytes / 1024**2:8.2f} MB")              # name and size in MB

print(f"\nWhole adapter folder: {total_bytes / 1024**2:.1f} MB")

In [ ]:
# =============================================================================
# CELL: 11.2 — LOAD THE ADAPTER ONTO A FRESH QWEN3 AND TEST IT
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# 1. Loads a new, untouched Qwen3-1.7B in 4-bit (as in Part 3.3).
# 2. Attaches the adapter saved in 11.1.
# 3. Runs one request through the full assistant with this reloaded model.
#
# If its decision matches what the trained model decided for the same
# request in Part 10, the save and load worked.
#
# This loads a second copy of Qwen3 (roughly 1.3 GB more GPU memory). The
# T4 has room for it.
# =============================================================================


fresh_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
reloaded_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)  # attach the saved A and B tables
reloaded_model.eval()                                               # answering mode

run_assistant("Start a SIP of 5k in SBI Credit Risk Fund on the 7th of every month", model=reloaded_model)

In [ ]:
# =============================================================================
# CELL (OPTIONAL): 11.3 — DOWNLOAD OR UPLOAD THE ADAPTER
# =============================================================================
#
# WHAT THIS CELL DOES
# -------------------
# Nothing, until you remove the "# " at the start of the code lines you want.
#
# OPTION A: download a zip to your laptop (the first 3 code lines).
# OPTION B: upload to your Hugging Face account (the last 3 code lines).
#           You need a free account and an access token with WRITE permission
#           (huggingface.co -> Settings -> Access Tokens), and must replace
#           YOUR_USERNAME with your username.
#
# For a seminar, do this once the day before. If live training is slow, load
# the saved adapter with the code in 11.2 and continue from Part 9.
# =============================================================================


# ----- OPTION A: download a zip -----
# import shutil                                          # built-in library for zipping folders
# shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)   # creates qwen3-mf-tool-calling-lora.zip
# from google.colab import files; files.download(ADAPTER_DIR + ".zip")  # starts the browser download

# ----- OPTION B: upload to Hugging Face -----
# from huggingface_hub import login; login()                      # paste your token when asked
# model.push_to_hub("YOUR_USERNAME/qwen3-mf-tool-calling-lora")    # upload the adapter
# tokenizer.push_to_hub("YOUR_USERNAME/qwen3-mf-tool-calling-lora")  # upload the tokenizer

---
# Part 12 — Recap

## What we built

A small model that acts as the **decision-maker** of a mutual fund assistant:

1. **Three tools** in plain Python: `get_nav` reads real NAVs from AMFI; `get_portfolio` and `start_sip` use demo data.
2. **Tool descriptions** in JSON-schema format: the only part of the tools the model ever sees.
3. **Generated training data:** sentence patterns filled with real AMFI fund names. Checking examples used new wording and new fund names.
4. **Training only on the answer:** labels of `-100` hid the long, repeated prompt from the loss.
5. **QLoRA:** 4-bit Qwen3-1.7B, frozen, with about 17.4 million trainable LoRA numbers (about 1%).
6. **Before/after scores** on 6 measures, and a **live assistant** that decides, calls the tool and replies.

## The key idea to remember

> **The model decides. Our code does.**
> Fine-tuning teaches the decision and the format. Facts such as today's NAV come from tools, live, and never from training.

## What we did not do

| Not done | Why it matters |
|---|---|
| Train the reply step (Part 10, step 4) | We relied on Qwen3's existing skill. A production system would include tool-result examples in training too |
| Multiple tool calls in one turn | For example, "NAV of A and B". The model can call tools several times; our data never shows this |
| Score the wording of text answers | We scored behaviour only (tool or no tool, asks or not) |
| Real customer data or real orders | `get_portfolio` and `start_sip` are demos. Real versions need authentication and a confirmation step before money moves |
| Handle a fund name that matches nothing | `get_nav` returns an error; a better assistant would suggest close matches |

## Words you now know

| Word | Meaning |
|---|---|
| Tool / function calling | The model writes which function to run and with which values; our code runs it |
| Tool description (JSON schema) | Name, purpose and values of a tool, as the model sees it |
| Arguments | The values a tool needs, such as `fund_name` or `amount` |
| `<tool_call>` | The tags Qwen3 wraps a tool call in |
| Thinking mode | Qwen3's optional step of writing its reasoning first; switched off here |
| Synthetic data | Training data generated by code from patterns, not collected from real users |
| Held-out patterns | Wording kept only for checking, so the score measures real learning |
| Labels / −100 | Which tokens the model learns from; −100 means "ignore this token" |
| Adapter | The saved LoRA tables |

## Exercises

1. **Add a fourth tool**, for example `redeem_units(fund_name, units)`. Add its description to `TOOLS`, write patterns for it in Part 4.2, add it to `KIND_MIX`, and retrain.
2. **Fewer examples.** Set `NUM_TRAIN_EXAMPLES` to 400. How much do the scores drop?
3. **Harder checking data.** Add checking patterns with typos (*"wat is the nav of…"*) or two values in an unusual order. Does the trained model still cope?
4. **Two tools in one request.** Write patterns like *"NAV of A and of B"* with two `tool_calls` in the correct answer. What changes in `full_conversation` and in the scoring?
5. **Smaller model.** Change `MODEL_ID` to `Qwen/Qwen3-0.6B`. Compare the before/after scores and the training time.